# Sums LULUCF GHG fluxes by feature in shapefiles

# Purpose: 
Calculate zonal statistics for LULUCF fluxes and their areas (vegetation, mineral soil, organic soil, total LULUCF) by contextual layers for shapefiles

# Inputs:
1. Path to local shapefile
2. Field in shapefile you want to use as the name for each feature
3. Other fields in the shapefile you want to carry through the analysis
4. S3 paths to zarrs of GHG fluxes from vegetation, mineral soil, organic soil, and total LULUCF
5. S3 paths to zarrs of contextual layers (currently vegetation model land state and starting composite primary forest)
6. Name of output xlsx
7. Metadata for xlsx

All of these can be adjusted in the "Run parameters" cell.

# Outputs (in a local folder):
1. csvs with raw/unformatted stats for every batch (in case processing fails part way through, this saves progress; job can be re-run from failed batch)
2. Consolidated and formatted stats by shapefile feature in an xlsx, with another tab comparing summed components with the LULUCF output (for QC).
   The xlsx has flux and relevant area for each component by year and contextual layer, as well as the area of each feature calculated as a polygon and as a raster (for QC).
   Only vegetation datasets are analyzed by land state node (since that doesn't mean anything for the soil datasets or total LULUCF). 

# Approach
This parallelizes zonal statistics calculation using xarray and Dask in Coiled clusters. The Coiled cluster is created part way through the notebook, just before it's actually used. 
That way, the cluster isn't sitting unused during preliminary steps.

**Batching:**
Analysis layers (flux layers) and contextual layers are read from zarrs in s3. 
Features in the shapefile are rasterized, essentially tile by tile. Features are also exploded (each part is processed separately, then stats are combined at the end) to speed up processing. 
The analysis is parallelized by running sets of 10x10 deg tiles in batches, with batches having 1 or more tiles. 
Batches are essentially determined by how many 1x1 deg chunks are needed for each tile (major determinant of Dask workload here), with the goal being that each batch is similar in complexity.
The reason to run in batches rather than a single analysis is that as the analysis gets larger, the preliminary Dask graph construction takes longer. Running in batches may slow down the analysis
compared to a global run but it also means that there will never be a giant Dask graph created up front that takes forever to send to the scheduler while the cluster is running.
The reason to run in batches with potentially multiple tiles vs. one tile at a time is that tiles that have geographically limited features in them will run very quickly and not use the cluster well, 
which isn't efficient.
Creating batches of a standardized complexity somewhat optimizes for the variety in feature distribution, size, and complexity within and between shapefiles 
(rather than saying batches will always have, e.g., 2 tiles). Within each batch, the Dask graph/task collection is created by iterating through the relevant individual tiles. 

**Handling overlaps between features:**
In this notebook, areas that are within multiple features are explicitly included in all relevant features, so this can result in double-counting if fluxes are summed without regard for overlapping areas.
The approach for dealing with overlapping areas is basically that sets of features in a tile that don't overlap are rasterized together as another "layer" of features. 
Thus, the more overlaps there are, the more separate rasterizations need to occur and the longer the processing takes. 
From working with BEF and two UNESCO files, the most layers in a tile was 8 (i.e. 8 xarray_reduce zonal stats passes were needed beyond a universal first pass to deal with all overlaps). 

**What is actually analyzed**
There are several outputs from this zonal stats notebook: vegetation, mineral soil, organic soil, and total LULUCF fluxes; the area of each of those (used for calculating 
flux per hectare); the area of each feature, according to raster analysis. Collectively, this is actually a lot of data to analyze and it can take quite a while. 


Per Claude session 'GHG flux calculation from shapefiles'

In [1]:
import geopandas as gpd
import xarray as xr
import rioxarray  
import coiled
import pandas as pd
import dask
import time
import sys
from pathlib import Path
from dask.distributed import Client, get_task_stream   # get_task_stream (not performance_report) reports
                                                          # actual-vs-expected task counts per batch --
                                                          # performance_report was tried first but failed on
                                                          # bokeh/jinja2 version mismatches; get_task_stream
                                                          # returns plain data with no rendering dependency
import numpy as np
import rasterio
import rasterio.features
from flox.xarray import xarray_reduce
import math
import shapely
from shapely.geometry import box
from shapely import wkb as shapely_wkb
from shapely.validation import explain_validity, make_valid
from shapely.geometry import GeometryCollection, MultiPolygon, Polygon
import numpy as np
from affine import Affine
import dask.array as da
from datetime import date
from flox import ReindexArrayType, ReindexStrategy
from pyproj import Geod
import os
from datetime import datetime

# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

pd.set_option('display.max_columns', None)

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu
from src.utilities import zonal_stats_utilities as zsu

In [2]:
### Run parameters

## Key parameters to change for each run

# Input shapefile: path and fields   
shapefile_path = "/mnt/c/GIS/Multi_project/brazil_biomes/bra_biomes.shp"   # Brazil biomes

# None: each feature is its own zone (identified by its FID / row number). Use when there's no unique ID field in the shapefile or every feature should have its own result. 
# Or name a shapefile column (e.g. "ID") to carry through to the results (if unique). Can also be used to merge features sharing a value into one zone (if not unique).
id_field = "Name"    

# optional: a shapefile column used only to label zones in the overlap report (e.g. "NAME_2"). None = no labels.
name_field = "Name"    

# Other fields in shp to include in output table
output_fields = []

# Base name for output spreadsheet
out_name = "Brazil_biomes"

# Determines whether the optional contextual layers are used
INCLUDE_LAND_STATE = True       # set False to skip the land_state contextual layer for this run
INCLUDE_PRIMARY_FOREST = True   # set False to skip the primary_forest contextual layer for this run

USE_GHG_SPECIFIC_EMISSIONS = False   # True: veg gross emissions split into CO2/CH4/N2O, organic soil emissions split into burned/drained by GHG (replaces the all-gases layers)

## Other parameters

# Input zarrs
veg_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_vegetation/version_1_0_5__standard__global/mega_zarr/annual_intervals/4000_pixels/20260130/vegetation_zarr.zarr/"
organic_soil_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/organic_soils/outputs/version_1_0_1/mega_zarr/ogh_mixed_f1_f15_f2_20260513/five_year/4000_pixels/20260525/mega.zarr/"  
min_soil_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_soil_organic_carbon/version_1_0_1__standard__global/zarr/4000_pixels/20260611/SOC_zarr.zarr/"  
lulucf_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs_LULUCF_totals/LULUCF_version_1_0_0_standard__global__veg_v1_0_5__org_soil_v1_0_1__min_soil_v1_0_1/zarr/annual_intervals/4000_pixels/20260614/LULUCF_annual.zarr/"
years = cn.veg_outputs_years  # [2016..2024], indexed 0-8 in the zarr's year dimension

primary_forest_zarr_path = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/starting_composite_primary_forest/version_1_0_5__standard__global/2015/presence/zarr/4000_pixels/20260210/starting_composite_primary_forest.zarr/"

# Output csv from each batch and the final xlsx go in this folder. 
# Timestamped so a run never overwrites a previous run.
# the batch loop writes one CSV per (batch, result-list) pair into this folder as it goes, so a
# crash partway through a long run can be resumed from the last completed batch instead of
# restarting from scratch.
run_folder = f"/mnt/c/GIS/AFOLU_flux_model/shapefile_zonal_stats/{out_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

today = date.today().strftime("%Y-%m-%d")
today_sparse = date.today().strftime("%Y%m%d")

readme_description = "GHG flux zonal statistics for UNESCO World Heritage site buffers"
readme_lines = [
    "Zonal statistics from Global Nature Watch's Land GHG Monitoring System: vegetation and soil flux data only",
    "CAUTION: DATA ARE PRELIMINARY AND UNPUBLISHED; DO NOT DISTRIBUTE WITHOUT PRIOR APPROVAL FROM CREATOR",
    "Created by David Gibbs",
    "david.gibbs@wri.org",
    today,
    readme_description,
    f"Shapefile used: {shapefile_path}",
    f"Years covered: {cn.veg_outputs_years[0]}-{cn.LC_last_year} ({cn.veg_end_year_count} years)",
    # f"Vegetation model: v{cn.veg_model_version}; time step: annual",
    f"Vegetation model: v1.0.5; time step: annual",  # Using this because the repo has moved on to v1.0.6 but I'm still using veg v1.0.5 for this analysis
    f"Organic soil model: v{cn.organic_soil_model_version}; time step: 5 years (2016-2020, 2021-2024)",
    f"Mineral soil model: v{cn.SOC_model_version}; time step: static (same value all years)",
    f"LULUCF model: v{cn.LULUCF_model_version}; time step: annual, with soil datasets annualized to match vegetation",
    f"Vegetation zarr: {veg_zarr_path}",
    f"Organic soil zarr: {organic_soil_zarr_path}",
    f"Mineral soil zarr: {min_soil_zarr_path}",
    f"LULUCF zarr: {lulucf_zarr_path}",
    f"Contextual layers included: land_state={INCLUDE_LAND_STATE}, primary_forest={INCLUDE_PRIMARY_FOREST}",
    f"Emissions by individual GHG (veg gross emissions and organic soil): {USE_GHG_SPECIFIC_EMISSIONS}",
] 
if INCLUDE_LAND_STATE:
    readme_lines.append(f"  Land state zarr is the same as the vegetation zarr: {veg_zarr_path}")
    readme_lines.append(f"  Land state lookup table: {cn.state_node_lookup_table_s3}")
if INCLUDE_PRIMARY_FOREST:
    readme_lines.append(f"  Primary forest zarr: {primary_forest_zarr_path}")
readme_lines += [
    "",
    "Emissions/net sources are positive; removals/net sinks are negative.",
    "land_state_node, land_state_meaning, and derived columns are just for vegetation data; they do not apply to soil or LULUCF data",
    "primary_forest is as defined in LULUCF manuscript; it is a combination of humid tropical primary forest, intact forest landscapes outside the tropics, and forest >100 years old.",
    "Calculated using flox zonal stats in a Coiled cluster" 
]

### Shapefile processing

In [3]:
%%time

### Reads shapefile and creates numeric identifying field

# Reads shapefile
print(f"Reading shapefile: {uu.timestr()}")

gdf = gpd.read_file(shapefile_path, encoding="utf-8")
print(f"Original CRS: {gdf.crs}: {uu.timestr()}")

if gdf.crs is None:
    raise ValueError("Shapefile has no CRS defined (missing/invalid .prj) -- can't safely reproject.")
elif gdf.crs.to_epsg() != 4326:
    print(f"Reprojecting from {gdf.crs.to_epsg()} to EPSG:4326")
    gdf = gdf.to_crs(epsg=4326)
else:
    print("Already EPSG:4326 -- no reprojection needed")

print(gdf.crs, len(gdf), "features")
print(gdf.columns)

if id_field is None:  # No ID field in shapefile specified. Creates geodataframe column with unique field for the rest of the analysis.  
    # Row position in file order = the GDAL/ArcGIS FID for an unfiltered read. Unique by construction,
    # works for any vector format, and needs no knowledge of the shapefile's attribute fields.
    gdf["source_fid"] = np.arange(len(gdf), dtype="uint32")
    gdf["raster_id"] = gdf["source_fid"]

else:  # ID field in shapefile specified. Runs analysis on this, regardless of whether the values are unique or not.     
    # Blank/missing IDs must be caught before factorizing: pd.factorize codes a missing value as -1, and
    # casting -1 to uint32 wraps to 4,294,967,295 -- exactly NO_ZONE_ID, the rasterizer's "no zone"
    # background value. A feature with a blank ID would silently vanish from the results.
    n_missing = gdf[id_field].isna().sum()
    if n_missing:
        raise ValueError(f"{id_field} has {n_missing:,} missing values -- fix them or choose a different id_field")

    # Converts the ID values (numeric or text, e.g. "USHa1980") to small whole numbers, which is what
    # rasterization/grouping needs. Codes are assigned in order of first appearance, starting at 0.
    gdf["raster_id"] = pd.factorize(gdf[id_field])[0].astype("uint32")
    gdf["source_fid"] = np.arange(len(gdf), dtype="uint32")   # still carried so outputs can be joined back

    n_zones = gdf["raster_id"].nunique()
    if n_zones != len(gdf):
        print(f"NOTE: {id_field} has {n_zones:,} unique values for {len(gdf):,} features -- "
              f"features sharing a value are treated as ONE zone")
field = "raster_id"

# Always carry the user's real ID / name columns into the output table, so a numeric
# raster_id never appears without its human-readable label.
for f in (name_field, id_field):
    if f is not None and f not in output_fields:
        output_fields = [f] + output_fields

print(f"Checking field names: {uu.timestr()}")
missing = [f for f in [field] + ([name_field] if name_field else []) + output_fields if f not in gdf.columns]
if missing:
    raise ValueError(f"Field(s) not found in shapefile: {missing}. Available columns: {list(gdf.columns)}")
print(f"Field names valid: {uu.timestr()}")

Reading shapefile: 20261014_14_18_03
Original CRS: EPSG:4326: 20261014_14_18_04
Already EPSG:4326 -- no reprojection needed
EPSG:4326 6 features
Index(['OBJECTID', 'Name', 'Shape_Leng', 'Shape_Area', 'geometry'], dtype='object')
Checking field names: 20261014_14_18_04
Field names valid: 20261014_14_18_04
CPU times: user 80.2 ms, sys: 30.4 ms, total: 111 ms
Wall time: 238 ms


In [4]:
%%time

### Identifies the shapefile's features that intersect each tile to feed the geometric validity-check cell right after it

# It gets rebuilt from scratch twice more downstream (once after geometry repair, once after multipolygon
# exploding), each time because the thing features_by_tile is built FROM has changed. Don't treat
# this particular features_by_tile as authoritative for anything past the geometry validity check.
# Overall, we are group features by tile_id because the script will iterate by tile or groups of tiles,
# so this is establishing the unit of processing. 

def build_features_by_tile(gdf):
    minx, miny, maxx, maxy = gdf.total_bounds
    west_edges = range(int(math.floor(minx / 10) * 10), int(math.ceil(maxx / 10) * 10), 10)
    north_edges = range(int(math.floor(miny / 10) * 10) + 10, int(math.ceil(maxy / 10) * 10) + 10, 10)
    candidate_tile_ids = [uu.xy_to_tile_id(west, north) for west in west_edges for north in north_edges]

    features_by_tile = {}
    for tile_id in candidate_tile_ids:
        tminx, tminy, tmaxx, tmaxy = uu.get_10x10_tile_bounds(tile_id)
        subset = gdf[gdf.geometry.intersects(box(tminx, tminy, tmaxx, tmaxy))]
        if len(subset):
            features_by_tile[tile_id] = subset
    return features_by_tile

features_by_tile = build_features_by_tile(gdf)

print({tid: len(sub) for tid, sub in features_by_tile.items()})

{'10S_080W': 1, '00N_080W': 1, '10N_080W': 1, '10S_070W': 2, '00N_070W': 1, '10N_070W': 1, '30S_060W': 1, '20S_060W': 4, '10S_060W': 4, '00N_060W': 2, '10N_060W': 1, '20S_050W': 3, '10S_050W': 3, '00N_050W': 3, '10N_050W': 1, '10S_040W': 2, '00N_040W': 2}
CPU times: user 20.8 ms, sys: 8.54 ms, total: 29.3 ms
Wall time: 27.6 ms


In [5]:
%%time

### Checks validity of features being processed

# Purely diagnostic -- doesn't change gdf or features_by_tile. Surfaces which features are invalid
# (self-intersecting rings, bowties, etc.) and why, before the next cell actually repairs them.
# Worth reading the reasons before repairing: make_valid's fix isn't always what you'd expect for
# every failure mode.

# The original per-tile listing prints every feature ID and validity flag for every tile. With hundreds
# of thousands of features that is an enormous amount of output, so it is off by default.
PRINT_PER_TILE_DETAIL = False

# --- Summary over the whole shapefile (each feature counted once) ---
# A feature that straddles tile boundaries appears in several tiles in features_by_tile, so counting
# per tile would double count it; counting on gdf itself does not.
geoms = gdf.geometry.to_numpy()
is_missing = shapely.is_missing(geoms)                 # null geometry
is_empty = shapely.is_empty(geoms)                     # present but has no coordinates
invalid_mask = ~shapely.is_valid(geoms) & ~is_missing  # fails topology rules (null geometries counted separately)

n = len(gdf)
n_invalid = int(invalid_mask.sum())

# Reasons are only computed for the invalid features -- skipping the valid ones saves a lot of time.
# shapely appends the coordinates of the problem point in brackets, e.g. "Self-intersection[10.2 5.1]";
# stripping them groups features by the TYPE of problem.
reasons = shapely.is_valid_reason(geoms[invalid_mask])
reason_type = pd.Series(reasons, dtype="object").str.replace(r"\[.*$", "", regex=True)

print(f"\n=== Geometry validity summary: {n:,} features ===")
print(f"Valid:                {n - n_invalid - int(is_missing.sum()):,}")
print(f"Invalid:              {n_invalid:,} ({n_invalid / n:.3%})")
print(f"Missing (null):       {int(is_missing.sum()):,}")
print(f"Empty:                {int(is_empty.sum()):,}")

if n_invalid:
    bad = pd.DataFrame({
        "reason": reason_type.to_numpy(),
        field: gdf.loc[invalid_mask, field].to_numpy(),
    })
    summary = (bad.groupby("reason")[field]
                  .agg(n_features="size", examples=lambda s: ", ".join(str(v) for v in s.head(3)))
                  .sort_values("n_features", ascending=False))
    print("\nInvalid features by reason (examples are the first 3 values of the zone ID field):")
    print(summary.to_string())

print("\nGeometry types:")
print(gdf.geom_type.value_counts().to_string())

# --- Original per-tile detail (optional) ---
if PRINT_PER_TILE_DETAIL:
    for tile in features_by_tile:
        print("tile_id:", tile)
        subset = features_by_tile[tile]
        print(field, subset[field].tolist())
        print("is_valid:", subset.geometry.is_valid.tolist())
        print("is_empty:", subset.geometry.is_empty.tolist())
        print("geom_type:", subset.geometry.geom_type.tolist())
        print("bounds:\n", subset.geometry.bounds)

        for _, row in subset[~subset.geometry.is_valid].iterrows():
            print(f"{field} {row[field]}: {explain_validity(row.geometry)}")

        print("\n")


=== Geometry validity summary: 6 features ===
Valid:                6
Invalid:              0 (0.000%)
Missing (null):       0
Empty:                0

Geometry types:
Polygon         4
MultiPolygon    2
CPU times: user 6.45 ms, sys: 6.2 ms, total: 12.6 ms
Wall time: 11.7 ms


In [6]:
%%time

### Fixes feature geometries and rebuilds feature lists for each tile for processing

# Repairs geometries
def repair(geom):
    if geom.is_valid:
        return geom
    fixed = make_valid(geom)  # A shapely function
    # make_valid on a genuinely broken polygon (self-intersection, bowtie, etc.) can decompose it
    # into a GeometryCollection containing a mix of geometry types (e.g. the real polygon area plus
    # a stray Point or LineString left over from the self-intersection). Only Polygon/MultiPolygon
    # parts are usable for zonal stats, so this filters down to just those and drops the rest.
    if isinstance(fixed, GeometryCollection):
        print(f"Fixing {GeometryCollection}")
        polys = [g for g in fixed.geoms if isinstance(g, (Polygon, MultiPolygon))]
        fixed = MultiPolygon(polys) if len(polys) > 1 else (polys[0] if polys else fixed)
    return fixed

print("Fixing geometries")

# QC: a correct topology repair should barely change a feature's area (it's fixing
# self-intersections, not redrawing the boundary). A large relative area change here would mean
# make_valid did something more drastic than expected and is worth a manual look.
areas_before = gdf.geometry.area.copy()
gdf["geometry"] = gdf.geometry.apply(repair)  # Geometry repairs are applied
areas_after = gdf.geometry.area
rel_diff = (areas_after - areas_before).abs() / areas_before.replace(0, np.nan)

print("All valid now:", gdf.geometry.is_valid.all())
print("Max relative area change:", rel_diff.max())

changed = rel_diff[rel_diff > 1e-6]
if len(changed):
    print("Features whose area changed by more than a negligible amount:")
    print(gdf.loc[changed.index, [field]].assign(rel_area_change=changed))
else:
    print("No feature's area changed by more than a negligible amount -- safe to proceed.")

# Rebuilds features_by_tile from the repaired gdf -- the previous version was built from the
# pre-repair geometries, so it would still hold the invalid ones otherwise.
features_by_tile = build_features_by_tile(gdf)

print({tid: len(sub) for tid, sub in features_by_tile.items()})
print("All valid across rebuilt features_by_tile:", all(sub.geometry.is_valid.all() for sub in features_by_tile.values()))

# for tile_id, subset in features_by_tile.items():
#     print(tile_id, subset.geometry.is_valid.tolist())

Fixing geometries
All valid now: True
Max relative area change: 0.0
No feature's area changed by more than a negligible amount -- safe to proceed.
{'10S_080W': 1, '00N_080W': 1, '10N_080W': 1, '10S_070W': 2, '00N_070W': 1, '10N_070W': 1, '30S_060W': 1, '20S_060W': 4, '10S_060W': 4, '00N_060W': 2, '10N_060W': 1, '20S_050W': 3, '10S_050W': 3, '00N_050W': 3, '10N_050W': 1, '10S_040W': 2, '00N_040W': 2}
All valid across rebuilt features_by_tile: True
CPU times: user 266 ms, sys: 32.8 ms, total: 299 ms
Wall time: 338 ms


<timed exec>:23: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

<timed exec>:25: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



In [7]:
### Explode MultiPolygon features into their individual disjoint parts before tile assignment.

# Exploding features into individual components saves computing time. 
# The way this zonal stats process works, the bounds of each feature are calculated, and all chunks
# within thouse bounds are used. Thus, a multipolygon that spans a very large area will include
# lots of chunks that all need to be computed together.
# Separating all parts of a feature into their own features means that each one can be processed
# separately in a smaller area.
# This arose from the UNESCO shapefiles, where some features are MultiPolygons whose parts are scattered across huge combined
# bounding boxes -- some are genuine transnational/serial sites, others are the shapefile data bug
# (a real site plus a bogus fragment in Mexico). Tile assignment already lets a
# multi-part feature appear in every tile any part touches, but the per-tile processing window is
# computed from subset.total_bounds -- the union bbox of every FEATURE (not part) assigned to that
# tile, clamped to the tile's own bounds. So a feature with a far-flung part still pushes the CURRENT
# tile's window out to the tile boundary in whatever direction that far-flung part lies, even if the
# local part is tiny -- every tile that happens to hold even one part of a globally-scattered feature
# gets forced to a full 100-chunk window. Exploding first means each tile's subset only ever contains
# the LOCAL part(s) actually inside it, so its window/weight reflects only what's really there.
# Validated on a related shapefile (World Heritage site buffers): exploding cut total batch-loop weight by ~21%
# with feature-overlap counts essentially unchanged (371 features flagged overlapping before AND
# after, despite the row count nearly tripling from exploding) -- i.e. it's a pure performance win,
# not a correctness change.

gdf["geometry_original"] = gdf.geometry  # Field used to calculate the pre-explosion area of each feature in a later cell

# Number of features before and after explosion
# All fields are copied into exploding features except geometry. Thus, there are no uniquely identifying fields at this point,
# which allows grouping of outputs by the shared original identifying field.
# However, it also requires care for merging/joining with the gdf from now on, since there are potentially copies of each row (for all polygon parts). 
n_before = len(gdf)
gdf = gdf.explode(index_parts=False).reset_index(drop=True)
gdf = gdf[gdf.geometry.area > 0].reset_index(drop=True)  # drop zero-area slivers introduced by exploding
n_after = len(gdf)
print(f"Exploded {n_before} features into {n_after} individual parts ({n_after - n_before} parts added)")

# Rebuild tile assignment using the exploded parts
features_by_tile = build_features_by_tile(gdf)

print(f"{len(features_by_tile)} tiles have features after exploding (rebuilt features_by_tile)")

Exploded 6 features into 50 individual parts (44 parts added)
17 tiles have features after exploding (rebuilt features_by_tile)


/tmp/ipykernel_50763/3103537236.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf = gdf[gdf.geometry.area > 0].reset_index(drop=True)  # drop zero-area slivers introduced by exploding


In [8]:
%%time

### Identifies which features spatially overlap which others and creates groupings of non-overlapping features into analysis "layers" accordingly

# rasterize() keeps one zone ID per pixel -- where two zones' geometries overlap, whichever is
# listed later in `shapes` wins every shared pixel and any earlier zones lose that area. 
# Thus, basic use of rasterize() doesn't allow area to be assigned to multiple, overlapping features in the same run.
# This cell addresses that limitation.

## Identifies all overlaps up-front (cheap, geometry-only, no cluster needed). 
# Overlaps are essentially separately processed as additional layers from non-overlapping areas. 
# However, not every overlap is worth the cost of an individual correction pass; a zone that loses a
# negligible sliver to a neighbor doesn't need one. OVERLAP_CORRECTION_THRESHOLD sets how much of a
# zone's OWN area has to be covered by another zone before it's worth correcting. 
# This is asymmetric: for a given pair of features, one zone can cross the threshold while the other doesn't;
# each zone's need for correction is judged against its own area only.
geod = Geod(ellps="WGS84")

OVERLAP_CORRECTION_THRESHOLD = 0.01  # Must have >1% overlap to be considered for separate analysis

overlap_partner_ids = {}   # dict: raster_id -> set of raster_ids of all other features it overlaps
                             # with, in either direction (built symmetrically below). This is the
                             # adjacency list the graph-coloring step later reads to decide which
                             # features can never share a correction layer.

overlap_needs_correction_ids = set()   # set of raster_ids for features where at least one overlap
                                         # exceeds OVERLAP_CORRECTION_THRESHOLD -- i.e. features that
                                         # actually need a dedicated correction layer, as opposed to
                                         # just accepting whatever the rasterizer's default tie-break
                                         # gives them. This is the set that drives which features get
                                         # pulled into tile_layers, and later which "base pass" rows
                                         # get dropped and replaced with corrected values in the table completion cell.

overlap_records = []   # list of one dict per detected overlapping pair (tile, both feature IDs,
                         # overlap area in ha, each side's fraction of its own area lost, whether it
                         # crossed the threshold). This is raw detail for the human-facing QC table
                         # (overlap_report) and the sensitivity printout only -- the actual correction
                         # logic doesn't read from it, only from overlap_partner_ids and
                         # overlap_needs_correction_ids above.

print(f"Starting overlap report: {uu.timestr()}")

# Iterates through tiles to check for overlaps between features
for tile_id, subset in features_by_tile.items():
    ids = subset[field].astype("uint32").values
    sindex = subset.sindex  # Spatial index, so that only nearby features are checked for overlaps
    geoms = subset.geometry.values
    for i, geom in enumerate(geoms):
        for j in sindex.query(geom, predicate="intersects"):
            # sindex.query returns every feature whose bounding box/geometry touches geom's, including
            # geom itself (j == i) and every pair it's already seen from the other side (j < i, found
            # earlier when the outer loop was at position j). Skipping both means each unordered pair
            # {i, j} only ever gets processed once, from whichever index is lower.
            if j <= i:
                continue
            inter_geom = geom.intersection(geoms[j])
            # predicate="intersects" is a coarse/fast pre-filter -- it can flag two features as
            # "intersecting" when they only touch along an edge or at a single point (shared boundary,
            # no shared interior). Those have zero actual overlap area and don't need correcting, so
            # this drops them before doing the more expensive geodesic area calculation below.
            if inter_geom.area == 0:
                continue
            # True earth-surface area of just the overlapping sliver (not the planar/degree-squared
            # area) -- needed because this value is reported directly in hectares in the QC table.
            # abs() because geod.geometry_area_perimeter can return a negative number depending on
            # which direction the polygon's ring winds.
            inter_area_ha = abs(geod.geometry_area_perimeter(inter_geom)[0]) * cn.m2_to_ha
            id_i, id_j = ids[i], ids[j]
            # Fractions use plain geometry.area (planar, in degrees^2), not the geodesic area above --
            # that's fine here because it's a ratio of two areas computed the same (biased) way, so the
            # projection distortion cancels out. Guards against dividing by zero for a degenerate
            # (zero-area) input geometry.
            frac_i = inter_geom.area / geom.area if geom.area > 0 else np.nan
            frac_j = inter_geom.area / geoms[j].area if geoms[j].area > 0 else np.nan

            # Records the overlap relationship in both directions (i knows about j, and j knows about
            # i) -- this symmetric adjacency map is what the later graph-coloring step reads to decide
            # which features are forbidden from sharing a correction layer.
            overlap_partner_ids.setdefault(id_i, set()).add(id_j)
            overlap_partner_ids.setdefault(id_j, set()).add(id_i)

            # Determines if overlap needs correction for features in pair (fraction of area lost is larger than threshold fraction).
            # Checked independently per side -- deliberately asymmetric (see cell-level comment above):
            # a tiny feature can cross the threshold against a huge neighbor even though that same
            # sliver is negligible from the huge neighbor's own point of view, and vice versa.
            if frac_i >= OVERLAP_CORRECTION_THRESHOLD:
                overlap_needs_correction_ids.add(id_i)
            if frac_j >= OVERLAP_CORRECTION_THRESHOLD:
                overlap_needs_correction_ids.add(id_j)

            # This dict is only for the human-readable QC table (overlap_report) built below -- it
            # plays no role in the actual correction logic, which relies solely on
            # overlap_partner_ids and overlap_needs_correction_ids. Every detected overlapping pair
            # gets one row here, whether or not it crossed the correction threshold, so the full
            # picture is visible for review.
            overlap_records.append({
                "tile_id": tile_id,
                field: id_i,
                "overlaps_with": id_j,
                "overlap_area_ha": inter_area_ha,
                "frac_of_this_feature": frac_i,
                "frac_of_other_feature": frac_j,
                "needs_correction": (frac_i >= OVERLAP_CORRECTION_THRESHOLD) or (frac_j >= OVERLAP_CORRECTION_THRESHOLD),
            })

# Starts preparing the feature overlap report (just for QC-- not used elsewhere in notebook)
overlap_report = pd.DataFrame(overlap_records)
# Guards against the case of zero overlaps found anywhere (e.g. a shapefile with no touching
# features) -- pd.DataFrame([]) on an empty list has no columns, so overlap_report[field] a few
# lines down would raise a KeyError without this check. In that case the report is just left empty.
if len(overlap_report):
    # drop_duplicates before set_index: after exploding features, gdf has multiple rows per `field`
    # value (one per part), so gdf.set_index(field) alone has a non-unique index and .map() raises
    # InvalidIndexError. Every part-row for a given ID carries the same Name, so taking the first is
    # always correct here, never an approximation.
    name_lookup = gdf.drop_duplicates(subset=field).set_index(field)[name_field]
    # Adds human-readable names purely for display -- the correction logic itself only ever uses the
    # numeric IDs.
    overlap_report[name_field] = overlap_report[field].map(name_lookup)
    overlap_report[f"overlaps_with_{name_field}"] = overlap_report["overlaps_with"].map(name_lookup)
    # Worst overlaps first, so a quick glance at the top of the displayed table shows the features
    # losing the largest share of their own area.
    overlap_report = overlap_report.sort_values("frac_of_this_feature", ascending=False)

# len(overlap_partner_ids) = how many distinct features are involved in at least one overlap at all
# (regardless of size); len(overlap_report) = how many overlapping pairs were found (a feature
# touching 3 neighbors contributes 3 rows here but counts once above).
print(f"{len(overlap_partner_ids)} of {len(gdf)} features overlap at least one other feature "
      f"({len(overlap_report)} overlapping pairs)")
print(f"{len(overlap_needs_correction_ids)} features need correction "
      f"(overlap covers >= {OVERLAP_CORRECTION_THRESHOLD:.1%} of their own area)")

# Sensitivity analysis of how many overlaps are included at different thresholds.
# Doesn't change analysis; it's just for reference. 
# Cheap to run because the expensive part (finding every overlapping pair and its area) already
# happened once, above; this just re-filters the already-computed frac_of_this_feature /
# frac_of_other_feature numbers in overlap_records against each candidate cutoff, with no new
# geometry operations. Lets you sanity-check that 1% is a reasonable cutoff rather than an arbitrary
# one, by seeing how sharply the correction-set size grows or shrinks around it.
# This would identify whether upping the overlap threshold would save much computing.
print("\nSensitivity of correction-set size to threshold:")
for t in [0.0, 0.001, 0.01, 0.05, 0.1, 0.25, 0.5, 0.9]:
    ids_at_t = set()
    for rec in overlap_records:
        if rec["frac_of_this_feature"] >= t:
            ids_at_t.add(rec[field])
        if rec["frac_of_other_feature"] >= t:
            ids_at_t.add(rec["overlaps_with"])
    print(f"  threshold >= {t:>6.1%}: {len(ids_at_t)} features would need correction")


## Designates combinations/groupings of non-overlapping features that are safe to analyze together in a single pass (called layers throughout (Claude's terminology))
# Dealing with overlapping features is probably the trickiest and most convoluted part of the analysis.
# The solution (after trying few other approaches) is essentially to group features in a tile that don't overlap each other
# into a set of layers, each of which is analyzed in separate, additional xarray_reduce passes beyond the base pass that all features in all tiles receive.
# Thus, this code just identifies groupings of overlapping features for additional passes.
# If no features overlap in a tile, there are 0 additional layers (all features analyzed together in the base xarray_reduce pass, no further passes to deal with overlaps).
# If there are overlaps in a tile, the features are divided up based on which ones don't overlap each other (didn't say that well). 
# Each of these combinations of features that constitutes a layer is analyzed in a separate xarray_reduce pass in a cell further down,
# so the more complicated the overlapping, the longer the analysis (xarray_reduce used more times). 

# Claude's description (for reference):
# Layer assignment: per tile, greedily color the correction-needing zones so no two zones
# sharing a layer overlap each other at all (uses ALL detected overlaps, not just above-threshold
# ones -- even a below-threshold overlap would still corrupt a shared rasterize pass). One combined
# rasterize+reduce per layer then replaces one-per-zone, at O(max overlap depth) cost per tile
# instead of O(affected zones). Welsh-Powell-style greedy coloring (highest-degree-first, assign the lowest
# unused color among neighbors) keeps the number of passes down to the tile's actual maximum overlap
# depth, which in practice has topped out around 5-8 layers even in tiles with dozens of affected zones.
#
# This graph-coloring (overlap handling) approach replaced an earlier per-zone individual-correction-pass design,
# which worked but blew up graph-build time badly once overlap correction was added at scale
# (~7.5 min to build one tile's graph, hundreds of MB) -- because it gave every affected zone its
# own dedicated rasterize+reduce pass regardless of how many zones actually needed to be separated
# from each other. 
# An alternative for dealing with overlaps could be to make each overlap area into its own feature and
# not what original features contributed to each overlap,
# then run the overlapping features along with all the original features, and 
# finally just add the fluxes for the overlapping features back to the relevant original ones at the end.
# Claude pointed out that the problem with that is the calculating all the overlaps in the first place
# could be very difficult and is sort of "fragile". I'm not going to try that.

tile_layers = {}   # dict: tile_id -> list of layers, each layer a list of zone IDs safe to
                     # rasterize together. This is the block's actual output, consumed by cell with xarray_reduce.
for tile_id, subset in features_by_tile.items():
    ids = subset[field].astype("uint32").values
    # Narrows down to just this tile's zones that are ALSO in the global correction set -- a zone
    # can be in this tile without needing correction (no meaningful overlap, or its overlap partner
    # lives in a different tile), and those are left out of the layering entirely.
    affected = [i for i in ids if i in overlap_needs_correction_ids]
    if not affected:
        # No zone in this tile needs correction -- zero layers, meaning zero extra passes for this
        # tile. The single base pass (built later, for every tile regardless of overlap)
        # is already correct on its own here.
        tile_layers[tile_id] = []
        continue
    affected_set = set(affected)
    # Local adjacency map, restricted to just this tile's correction-needing zones -- looks up each
    # zone's overlap partners in the global overlap_partner_ids map, but drops any partner that
    # either didn't cross the correction threshold or isn't in this tile. Coloring only needs to keep
    # correction-needing zones apart from EACH OTHER, not from every zone they happen to touch.
    adjacency = {i: {j for j in overlap_partner_ids.get(i, ()) if j in affected_set} for i in affected}

    layers_assigned = {}   # zone_id -> layer number, filled in one zone at a time below
    # Processes the most-connected zones first (Welsh-Powell greedy coloring) -- placing the
    # "hardest to fit" zones before their less-constrained neighbors tends to produce fewer total
    # layers than processing in an arbitrary order.
    # That is, ranks features by how connected they are (how many overlaps they have)
    for zone_id in sorted(affected, key=lambda z: -len(adjacency[z])):
        # Which layer numbers are already taken by this zone's neighbors that have been colored so
        # far. Neighbors not yet reached in this loop don't count yet -- their constraint against
        # zone_id gets enforced later, when THEY are processed and see zone_id already colored.
        used_by_neighbors = {layers_assigned[n] for n in adjacency[zone_id] if n in layers_assigned}
        # Picks the lowest-numbered layer not already used by any colored neighbor -- this greedy
        # choice is what keeps the total layer count low without an expensive search for the
        # mathematically optimal (fewest-possible) coloring.
        layer = 0
        while layer in used_by_neighbors:
            layer += 1
        layers_assigned[zone_id] = layer

    # Total layers needed for this tile = highest layer number used + 1 (layers are numbered from 0).
    n_layers = max(layers_assigned.values(), default=-1) + 1
    layers = [[] for _ in range(n_layers)]
    # Reshapes "zone -> layer number" into "layer number -> list of zones" -- the shape later cell
    # actually loops over (one rasterize+reduce per entry in this list).
    for zone_id, layer in layers_assigned.items():
        layers[layer].append(zone_id)
    tile_layers[tile_id] = layers

# Pure reporting from here down -- summarizes tile_layers for a human to review, doesn't feed back into any computation.
layer_counts_by_tile = {tile_id: len(layers) for tile_id, layers in tile_layers.items()}
print(f"\nLayering summary: max layers beyond the base pass needed in any single tile = {max(layer_counts_by_tile.values(), default=0)}")
print("Layer count by tile (tiles needing correction only, worst first):")
for tile_id, n in sorted(layer_counts_by_tile.items(), key=lambda x: -x[1]):
    if n > 0:
        print(f"  {tile_id}: {n} layers")

display(overlap_report)   # the raw overlap-pair table built in the first half of this cell, shown here for manual review

Starting overlap report: 20261014_14_18_04
0 of 50 features overlap at least one other feature (0 overlapping pairs)
0 features need correction (overlap covers >= 1.0% of their own area)

Sensitivity of correction-set size to threshold:
  threshold >=   0.0%: 0 features would need correction
  threshold >=   0.1%: 0 features would need correction
  threshold >=   1.0%: 0 features would need correction
  threshold >=   5.0%: 0 features would need correction
  threshold >=  10.0%: 0 features would need correction
  threshold >=  25.0%: 0 features would need correction
  threshold >=  50.0%: 0 features would need correction
  threshold >=  90.0%: 0 features would need correction

Layering summary: max layers beyond the base pass needed in any single tile = 0
Layer count by tile (tiles needing correction only, worst first):


""


CPU times: user 164 ms, sys: 0 ns, total: 164 ms
Wall time: 167 ms


In [9]:
%%time

### Creates batches of tiles (1 or more tiles) of roughly equal "weights"

# This establishes how many batches the tile will be run in. 
# Each batch involves the construction of a Dask graph, running tasks, and an output csv with the stats for that batch.
# Batch weight is basically based on the number of 4000x4000 pixel chunks needed (details below).
# It's better to set TARGET_WEIGHT_PER_BATCH than the number of batches because a given batch weight
# should work across different shapefiles; once a batch weight that works will on a given cluster size is 
# established, that batch weight can be used on other shapefiles (or further experimented with). 

TARGET_WEIGHT_PER_BATCH = 700   # Tune this, not N_BATCHES directly

# Determines how many 4000-pixel zarr chunks a bounding box spans. This is the proxy
# for a window's memory/compute cost, since the batch loop pulls whole chunks from S3 regardless of
# how much of each chunk the real geometry covers.
def estimate_chunks(minx, miny, maxx, maxy):
    w_px = (maxx - minx) / cn.resolution
    h_px = (maxy - miny) / cn.resolution
    return math.ceil(w_px / cn.chunk_dims) * math.ceil(h_px / cn.chunk_dims)

# Union bounding box of a layer's geometries, clamped to the tile's working window.
# Shared with the batch-weight estimate so the estimate and the actual processing
# window always agree.
def layer_bounds(layer_geoms, minx, miny, maxx, maxy, buffer):
    lminx = min(g.bounds[0] for g in layer_geoms)
    lminy = min(g.bounds[1] for g in layer_geoms)
    lmaxx = max(g.bounds[2] for g in layer_geoms)
    lmaxy = max(g.bounds[3] for g in layer_geoms)
    lminx = max(minx, lminx - buffer)
    lminy = max(miny, lminy - buffer)
    lmaxx = min(maxx, lmaxx + buffer)
    lmaxy = min(maxy, lmaxy + buffer)
    return lminx, lminy, lmaxx, lmaxy

# This tile-level weight formula went through three iterations before landing here:
#   1) len(ids) + 8*n_layers -- raw feature count as the cost proxy. Broke down in practice: one
#      tile (00N_080W) had MORE actual tasks than a tile the formula scored as heavier, because
#      feature count says nothing about how big a tile's rasterize WINDOW is -- a tile with few
#      features scattered edge-to-edge needs the full 10x10deg window just like a tile with many
#      tightly-clustered features.
#   2) n_chunks*(1+n_layers) -- switched to a chunk-count-based window size, which fixed the above,
#      but still charged every overlap-correction layer for the tile's FULL window even though a
#      layer's own zones might only occupy a small corner of it.
#   3) (current) -- sum of the base window's chunks, PLUS each layer's OWN clipped-window chunks
#      (each layer's bounding box is just its own zones, not the whole tile). This is also what
#      fixed a real KilledWorker memory crash: before per-layer clipping existed, every correction
#      layer's mask spanned the full tile regardless of how few zones it covered, and graph-build
#      time for one bad tile went from what should be seconds to ~7.5 minutes and hundreds of MB.
# Better weight metrics could be calculated but this has worked fine and I don't think it needs
# improvement at this point. 
# Batch weight is an entirely artificial metric; it's just supposed to approximate how complex
# the batch will be for Dask to handle. 
tile_weights = {}
for tile_id, subset in features_by_tile.items():
    tminx, tminy, tmaxx, tmaxy = uu.get_10x10_tile_bounds(tile_id)
    fminx, fminy, fmaxx, fmaxy = subset.total_bounds
    buffer = 0.001  # Expands the bounding box by the specified decimal degrees
    minx = max(tminx, fminx - buffer); miny = max(tminy, fminy - buffer)  # Clamps bounding box of features to buffered extent or tile's edges
    maxx = min(tmaxx, fmaxx + buffer); maxy = min(tmaxy, fmaxy + buffer)  # Clamps bounding box of features to buffered extent or tile's edges

    # Complexity weight for each tile
    weight = estimate_chunks(minx, miny, maxx, maxy)  

    ids = subset[field].astype("uint32").values
    geoms = subset.geometry.values
    for layer_ids in tile_layers.get(tile_id, []):
        layer_id_set = set(layer_ids)
        layer_geoms = [g for g, zid in zip(geoms, ids) if zid in layer_id_set]
        lminx, lminy, lmaxx, lmaxy = layer_bounds(layer_geoms, minx, miny, maxx, maxy, buffer)
        weight += estimate_chunks(lminx, lminy, lmaxx, lmaxy)

    tile_weights[tile_id] = weight

# Establishes how many batches are needed based on total weight and per-batch target weight
total_weight = sum(tile_weights.values())
N_BATCHES = max(1, min(len(tile_weights), math.ceil(total_weight / TARGET_WEIGHT_PER_BATCH)))
print(f"Total weight: {total_weight} across {len(tile_weights)} tiles -> N_BATCHES = {N_BATCHES}")

# Assigns tiles to batches.
# Longest-Processing-Time (LPT) greedy bin-packing: process tiles heaviest-first, always adding the
# next tile to whichever batch currently has the smallest running total. Simple, deterministic, and
# gives near-optimal balance for this problem size; chosen over more complex bin-packing algorithms
# because the marginal balance improvement wasn't worth the complexity once weight (not raw tile/task
# count) became the right cost signal.
tile_batches = [[] for _ in range(N_BATCHES)]
batch_totals = [0] * N_BATCHES
for tile_id in sorted(tile_weights, key=lambda t: tile_weights[t], reverse=True):
    lightest = batch_totals.index(min(batch_totals))
    tile_batches[lightest].append(tile_id)
    batch_totals[lightest] += tile_weights[tile_id]

print("Batch sizes (tiles, total weight):")
for i, (batch, total) in enumerate(zip(tile_batches, batch_totals)):
    preview = ", ".join(batch[:5])
    print(f"  Batch {i}: {len(batch)} tiles, weight {total} -- up to the first 5: {preview}")

Total weight: 1194 across 17 tiles -> N_BATCHES = 2
Batch sizes (tiles, total weight):
  Batch 0: 8 tiles, weight 588 -- up to the first 5: 10S_070W, 20S_060W, 00N_060W, 10S_050W, 10N_070W
  Batch 1: 9 tiles, weight 606 -- up to the first 5: 00N_070W, 10S_060W, 20S_050W, 00N_050W, 10N_060W
CPU times: user 4.91 ms, sys: 0 ns, total: 4.91 ms
Wall time: 4.36 ms


In [10]:
%%time

### Gets the area of each feature using geodesic area (reported in hectares)

def compute_area_ha(geom):
    area_m2, _ = geod.geometry_area_perimeter(geom)
    return abs(area_m2) * cn.m2_to_ha

print(f"Calculating feature areas from polygons: {uu.timestr()}")
# geometry_original is duplicated across every exploded part-row of a feature (one identical copy
# per part -- see cell 6), so running compute_area_ha on every row of gdf recomputes the same
# feature's geodesic area once per part instead of once per feature. Deduplicating to one row per
# feature first, then mapping the result back by field, cuts the number of (relatively expensive)
# geometry_area_perimeter calls from one-per-part down to one-per-feature.
unique_features = gdf.drop_duplicates(subset=field)
area_by_id = unique_features.set_index(field)["geometry_original"].apply(compute_area_ha)
gdf["feature_total_area_ha"] = gdf[field].map(area_by_id)
print(f"Calculated feature areas from polygons: {uu.timestr()}")

Calculating feature areas from polygons: 20261014_14_18_04
Calculated feature areas from polygons: 20261014_14_18_04
CPU times: user 27.6 ms, sys: 0 ns, total: 27.6 ms
Wall time: 26.4 ms


### Analysis and contextual layer inputs

In [11]:
### Analysis layers

# Mineral soil (uses change for 2020 only (avg(2010/2015) vs. avg(2015/2020)))
min_soil_year_idx = cn.SOC_change_intervals.index(2020) + 1
min_soil_var_names = {
    "mineral_soil_gross_loss_MgCO2": f"{cn.SOC_loss_min_soil_extent_pattern}_ha_yr",
    "mineral_soil_gross_gain_MgCO2": f"{cn.SOC_gain_min_soil_extent_pattern}_ha_yr",
    "mineral_soil_net_change_MgCO2": f"{cn.SOC_net_min_soil_extent_pattern}_ha_yr",
}
min_soil_ds = xr.open_zarr(min_soil_zarr_path, consolidated=False)
# Mineral soil is static (one value, not one per analysis year) -- selects the single relevant
# interval up front and drops the year dim, so it doesn't need special per-year alignment handling
# downstream the way organic soil does.
min_soil_layers_static = {
    label: min_soil_ds[var].isel(year=min_soil_year_idx).drop_vars("year")
        .rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in min_soil_var_names.items()
}

# Organic soil (five-year intervals: 2016_2020 and 2021_2024)
if USE_GHG_SPECIFIC_EMISSIONS:
    organic_soil_var_names = {
        # Burned
        "organic_soil_burned_ch4_Mg_CO2e": "burned_ch4_Mg_CO2e_ha_yr",
        "organic_soil_burned_co2_Mg_CO2": "burned_co2_Mg_CO2_ha_yr",
        # Drained
        "organic_soil_drained_ch4_ditch_Mg_CO2e": "drained_ch4_ditch_Mg_CO2e_ha_yr",
        "organic_soil_drained_ch4_land_Mg_CO2e": "drained_ch4_land_Mg_CO2e_ha_yr",
        "organic_soil_drained_co2_Mg_CO2": "drained_co2_Mg_CO2_ha_yr",
        "organic_soil_drained_co2_offsite_Mg_CO2": "drained_co2_offsite_Mg_CO2_ha_yr",
        "organic_soil_drained_n2o_Mg_CO2e": "drained_n2o_Mg_CO2e_ha_yr",
    }
    # Used in cell 14 to rebuild the standard fire/drainage subtotals from the by-GHG layers
    organic_burned_labels = [l for l in organic_soil_var_names if "_burned_" in l]
    organic_drained_labels = [l for l in organic_soil_var_names if "_drained_" in l]
else:
    organic_soil_var_names = {
        "organic_soil_emissions_fire_MgCO2e": cn.burned_organic_soils_total_pattern,
        "organic_soil_emissions_drainage_MgCO2e": cn.drained_organic_soils_total_pattern,
    }
organic_soil_ds = xr.open_zarr(organic_soil_zarr_path, consolidated=False)

# Fail fast if a requested variable isn't in the zarr, instead of erroring partway through a run
missing = [v for v in organic_soil_var_names.values() if v not in organic_soil_ds.data_vars]
if missing:
    raise ValueError(f"Organic soil zarr is missing variables: {missing}")

organic_soil_layers_native = {
    label: organic_soil_ds[var].isel(year=slice(-len(cn.organic_soil_year_intervals), None))
        .assign_coords(year=np.arange(len(cn.organic_soil_year_intervals)))
        .rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in organic_soil_var_names.items()
}

# Vegetation (annual)
if USE_GHG_SPECIFIC_EMISSIONS:
    veg_var_names = {
        "veg_gross_emissions_CO2_only_MgCO2": "gross_emissions__all_C_pools__CO2_only__MgCO2_ha_yr",
        "veg_gross_emissions_CH4_MgCO2e": "gross_emissions__CH4__MgCO2e_ha_yr",
        "veg_gross_emissions_N2O_MgCO2e": "gross_emissions__N2O__MgCO2e_ha_yr",
        "veg_gross_removals_MgCO2e": "gross_removals__all_C_pools__MgCO2_ha_yr",
        "veg_net_flux_MgCO2e": "net_flux__all_C_pools__all_gases__MgCO2e_ha_yr",
    }
    veg_gross_emission_labels = [l for l in veg_var_names if l.startswith("veg_gross_emissions_")]
else:
    veg_var_names = {
        "veg_gross_emissions_MgCO2e": "gross_emissions__all_C_pools__all_gases__MgCO2e_ha_yr",
        "veg_gross_removals_MgCO2e": "gross_removals__all_C_pools__MgCO2_ha_yr",
        "veg_net_flux_MgCO2e": "net_flux__all_C_pools__all_gases__MgCO2e_ha_yr",
    }
veg_ds = xr.open_zarr(veg_zarr_path, consolidated=False)

# Fail fast if a requested variable isn't in the zarr, instead of erroring partway through a run
missing = [v for v in veg_var_names.values() if v not in veg_ds.data_vars]
if missing:
    raise ValueError(f"Vegetation zarr is missing variables: {missing}")

veg_analysis_layers = {
    label: veg_ds[var].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in veg_var_names.items()
}

# LULUCF (annual)
lulucf_var_names = {
    "LULUCF_gross_emissions_MgCO2e": f"{cn.gross_emis_all_C_pools_all_gases_LULUCF_pattern}_ha_yr",
    "LULUCF_gross_removals_MgCO2e": f"{cn.gross_removals_all_C_pools_LULUCF_pattern}_ha_yr",
    "LULUCF_net_flux_MgCO2e": f"{cn.net_flux_all_C_pools_all_gases_LULUCF_pattern}_ha_yr",
}
lulucf_ds = xr.open_zarr(lulucf_zarr_path, consolidated=False)
lulucf_layers = {
    label: lulucf_ds[var].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    for label, var in lulucf_var_names.items()
}

# Annual datasets merged -- merges directly into the same stack/reduce, unlike mineral or organic soil. LULUCF flux is
# technically reconstructable by summing veg + soils, but LULUCF's area is NOT reconstructable from components'
# separate area totals the same way. Thus, LULUCF is kept as its own full analysis layer rather than
# dropped in favor of "just sum the components," even though that would be cheaper.
# If I didn't want the analysis area for LULUCF, I could just drop LULUCF and sum the components to get the same thing. 
annual_analysis_layers = {**veg_analysis_layers, **lulucf_layers}
var_names = {**veg_var_names, **lulucf_var_names, **min_soil_var_names, **organic_soil_var_names}

# Pixel area
pixel_area_ds = xr.open_zarr(cn.pixel_area_zarr_path, consolidated=False).rename_vars(band_data="pixel_area_ha")
pixel_area = pixel_area_ds["pixel_area_ha"].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")

# Resolution and dimensions of the actual analysis-ready layers (not the raw zarrs) 
# This should show
# mineral soil with no year dim (collapsed to one value) and organic soil with year=2 (sliced
# down from the raw zarr's 5), rather than the raw zarrs' native shapes.
min_soil_check = next(iter(min_soil_layers_static.values()))
organic_soil_check = next(iter(organic_soil_layers_native.values()))
veg_check = next(iter(veg_analysis_layers.values()))
lulucf_check = next(iter(lulucf_layers.values()))

print("mineral soil:  x res:", float(min_soil_check["x"][1] - min_soil_check["x"][0]), " size:", min_soil_check.sizes)
print("organic soil:  x res:", float(organic_soil_check["x"][1] - organic_soil_check["x"][0]), " size:", organic_soil_check.sizes)
print("vegetation  :  x res:", float(veg_check["x"][1] - veg_check["x"][0]), " size:", veg_check.sizes)
print("lulucf      :  x res:", float(lulucf_check["x"][1] - lulucf_check["x"][0]), " size:", lulucf_check.sizes)
print("pixel_area  :  x res:", float(pixel_area["x"][1] - pixel_area["x"][0]), " size:", pixel_area.sizes)

mineral soil:  x res: 0.0002499999999940883  size: Frozen({'y': 720000, 'x': 1440000})
organic soil:  x res: 0.0002499999999940883  size: Frozen({'year': 2, 'y': 720000, 'x': 1440000})
vegetation  :  x res: 0.0002499999999940883  size: Frozen({'year': 9, 'y': 720000, 'x': 1440000})
lulucf      :  x res: 0.0002499999999940883  size: Frozen({'year': 9, 'y': 720000, 'x': 1440000})
pixel_area  :  x res: 0.0002499999999940883  size: Frozen({'y': 560000, 'x': 1440000})


In [12]:
### Contextual layers (optional)

if INCLUDE_LAND_STATE:
    # land_state_node already lives inside the vegetation mega_zarr and has the same 9-year dimension
    # as the flux variables, so no special year-alignment handling is needed (unlike soil datasets).
    # land_state_da = veg_ds[cn.land_state_pattern].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")  # v1.0.6 pattern
    land_state_da = veg_ds['land_state_node'].rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y") # v1.0.5 pattern

    # Creates dataframe of land state nodes for joining to output stats table
    state_node_df = zsu.create_state_node_df(cn.state_node_lookup_table_local, cn.state_node_lookup_table_s3, cn.sheet)
    node_codes = np.array(list(state_node_df["land_state"]), dtype=np.uint32)
    print("land_state    x res:", float(land_state_da["x"][1] - land_state_da["x"][0]), " size:", land_state_da.sizes)
else:
    print("land_state contextual layer disabled for this run (INCLUDE_LAND_STATE=False)")

if INCLUDE_PRIMARY_FOREST:
    # starting_composite_primary_forest is a separate, static (single 2015 snapshot) zarr -- no year dimension
    primary_forest_ds = xr.open_zarr(primary_forest_zarr_path, consolidated=False)
    primary_forest_da = primary_forest_ds[cn.starting_composite_primary_forest_pattern]
    if "year" in primary_forest_da.dims:
        primary_forest_da = primary_forest_da.isel(year=0, drop=True)
    primary_forest_da = primary_forest_da.rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
    primary_forest_labels = {0: "not primary forest", 1: "primary forest"}
    print("prim_for      x res:", float(primary_forest_ds["x"][1] - primary_forest_ds["x"][0]), " size:", primary_forest_ds.sizes)
else:
    print("primary_forest contextual layer disabled for this run (INCLUDE_PRIMARY_FOREST=False)")

land_state    x res: 0.0002499999999940883  size: Frozen({'year': 9, 'y': 720000, 'x': 1440000})
prim_for      x res: 0.0002499999999940883  size: Frozen({'year': 1, 'y': 720000, 'x': 1440000})


### Zonal stats

In [13]:
# Spot instances are cheaper but can silently lose workers over a long run and never replace them.
# On-demand doesn't have this problem, at higher $/hr cost. N_BATCHES > 7 is used as a rough proxy for "this
# run is long enough that reliability matters more than the spot discount"; short/cheap runs can still risk spot.
if N_BATCHES > 7:
    worker_type = "on-demand"
else:
    worker_type = "spot_with_fallback"
print(f"Using {worker_type} workers")

cluster = coiled.Cluster( 
    n_workers=50,  
    use_best_zone=True, 
    compute_purchase_option=worker_type,
    idle_timeout="15 minutes",
    region="us-east-1",
    name="custom_AOI_zonal_stats__optional_contextual_layers",
    workspace='wri-land-research',
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r8g.2xlarge",
    worker_vm_types="r8g.2xlarge",
    # nthreads controls how many tiles' worth of rasterize+reduce work can run CONCURRENTLY on one
    # worker, which is what actually drives per-worker memory pressure. Confirmed via
    # distributed.nanny.memory log messages ("exceeded 95% memory budget. Restarting...") that the
    # default of 8 threads causes repeated worker restarts/crashes under this workload; nthreads=1
    # got past it but ran ~2-3x slower. 5 is the empirical middle ground landed on for this run --
    # worth revisiting (both directions) if either crashes recur or the run feels unnecessarily slow.
    worker_options={"nthreads": 5},
    **({'software': 'afolu_zonal_stats_20251222'}) 
)

client = cluster.get_client()

Using spot_with_fallback workers


Output()

/home/dagibbs22/miniforge3/envs/coiled_20260815/lib/python3.12/site-packages/distributed/client.py:1612: VersionMismatchWarning: Mismatched versions found

+-------------+--------+-----------+---------+
| Package     | Client | Scheduler | Workers |
+-------------+--------+-----------+---------+
| cloudpickle | 3.1.1  | 3.1.2     | 3.1.2   |
| msgpack     | 1.1.0  | 1.2.1     | 1.2.1   |
| toolz       | 1.0.0  | 1.1.0     | 1.1.0   |
| tornado     | 6.5.1  | 6.5.7     | 6.5.7   |
+-------------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [14]:
%%time

### Actual zonal stats calculations in Dask

# Creates the timestamped run folder if it doesn't already exist. This is where
# each batch's checkpoint CSVs get written below, so a crash partway through a long run can be
# resumed from the last completed batch instead of starting over.
os.makedirs(run_folder, exist_ok=True)
print(f"Saving batch outputs to: {run_folder}")

# Names of the distinct outputs that come from the zonal stats
STREAM_NAMES = [
    # Outputs from base pass (doesn't account for overlapping features). Outputs for features that overlap are discarded later. 
    "annual",                      # Annual datasets: vegetation and LULUCF (also intersected with primary forst and land state)
    "min_soil",                    # Mineral soil (1 time interval, intersected with primary forest) 
    "organic",                     # Organic soil (2 time intervals, intersected with primary forest)
    "rasterized_area",             # Total area of polygon as calculated in raster-land (to compare with polygon-based area for QC) (ha)
    "annual_data_area",            # Area of pixels with annual data (different for vegetation and LULUCF) to calculate flux/ha
    "min_soil_data_area",          # Area of pixels with mineral soil data to calculate flux/ha
    "organic_data_area",           # Area of pixels with organic soil data to calculate flux/ha
    # Outputs from subsequent 
    "overlap_annual",              # Same as above, but from subsequent passes from overlap layers
    "overlap_min_soil",            # Same as above, but from subsequent passes from overlap layers
    "overlap_organic",             # Same as above, but from subsequent passes from overlap layers
    "overlap_area",                # Area of overlap layers
    "overlap_annual_data_area",    # Same as above, but from subsequent passes from overlap layers
    "overlap_min_soil_data_area",  # Same as above, but from subsequent passes from overlap layers
    "overlap_organic_data_area",   # Same as above, but from subsequent passes from overlap layers
]
# One (tile_id, result) list per stream, keyed by name. Keying by name makes the correspondence
# structural instead of positional.
results = {}
for name in STREAM_NAMES:
    results[name] = []

# Running totals accumulated across every batch, printed in the final summary once the loop below
# finishes. expected vs actual task counts (from get_task_stream, further down) flag worker
# instability/retries even when a batch doesn't outright fail. duration vs compute_duration lets you
# see how much of each batch's wall-clock time was spent actually computing on the cluster versus
# building/introspecting the graph and writing checkpoint CSVs locally.
total_expected_tasks = 0
total_actual_tasks = 0
total_duration_sec = 0.0
total_compute_duration_sec = 0.0

# NO_ZONE_ID: rasterio.features.rasterize needs a "background" fill value for pixels no shape
# covers. Using fill=0 (the obvious default) silently collides with any real zone whose ID happens
# to be literally 0. This happened as a real bug on a shapefile that had a zone ID of 0: every
# unclaimed background pixel in that zone's tile got misattributed to it. 
# np.iinfo("uint32").max can never collide with a real zone code (raster_id is
# factorize-derived, always far smaller), so it's a safe universal sentinel regardless of what
# shapefile this notebook is pointed at.
NO_ZONE_ID = np.iinfo("uint32").max  # sentinel background value; can't collide with any real zone ID

# Rasterizes the provided features in the 1x1 deg chunk.
# It is used both to rasterize all features in the chunk in the base pass, 
# and also to rasterize the features in the subsequent overlapping features layers. 
# Defined at TOP LEVEL (not nested inside another function) deliberately -- a nested closure here
# failed to serialize across the Coiled cluster via cloudpickle (AttributeError/IndexError on the
# scheduler side, since cloudpickle can't reliably pickle a function's enclosing closure state for
# arbitrary nested defs). A plain top-level function with shapes/transform passed as explicit
# map_blocks kwargs pickles cleanly.
def rasterize_chunk(block, shapes, transform, block_info=None):
    (y0, y1), (x0, x1) = block_info[None]["array-location"]
    chunk_transform = transform * Affine.translation(x0, y0)
    return rasterio.features.rasterize(
        shapes, out_shape=(y1 - y0, x1 - x0), transform=chunk_transform,
        fill=NO_ZONE_ID, all_touched=False, dtype="uint32",
    )

# Rasterizes one overlap-correction layer's geometries onto whatever source grid template_2d comes
# from (veg/LULUCF, mineral soil, or organic soil -- called once per source, each with its own
# template/transform but the same shapes). template_2d's own values are never used, only its shape
# and dask chunk layout -- it exists purely so map_blocks knows how to size and chunk the output.
# name=field here is critical and was the source of a real, previously-silent data-loss bug: without
# it, the DataArray returned by this function has no name, so when it's later turned into a
# DataFrame and grouped by column name, the grouping column comes out as NaN -- and pandas' default
# groupby silently DROPS rows with a NaN key. That meant every overlap-corrected zone's contribution
# was computed correctly but then silently discarded before ever reaching the output table, in runs
# that otherwise looked completely successful (no error, no warning). One line (name=field) fixed it.
def rasterize_layer(template_2d, transform, shapes):
    # Runs rasterize_chunk once per dask chunk of template_2d.data, burning shapes into each chunk
    # using this window's transform. Still lazy -- nothing is actually rasterized until compute() runs.
    zone_dask = da.map_blocks(
        rasterize_chunk, template_2d.data,
        shapes=shapes, transform=transform, dtype="uint32",
    )
    # Wraps the raw dask array back into a labeled DataArray, copying template_2d's own y/x
    # coordinates so it lines up exactly with the source it was built from (no xr.align needed,
    # unlike the base pass's shared zone_da). name=field is required -- without it, downstream
    # groupby silently drops every pixel via a NaN grouping key.
    return xr.DataArray(zone_dask, dims=("y", "x"),
                         coords={"y": template_2d["y"], "x": template_2d["x"]}, name=field)

# Iterates through batches of tiles. This is the level at which the parallelization occurs.
# Resuming a job if a batch fails would happen here.
for batch_num, batch_tile_ids in enumerate(tile_batches):
    batch_start = time.time()
    print(f"\n=== Batch {batch_num + 1}/{N_BATCHES}: {len(batch_tile_ids)} tiles -- building graph: {uu.timestr()} ===")

    # Per-batch accumulator, reset fresh every iteration of the outer loop -- mirrors the shape of
    # the persistent results dict above, but only holds this batch's LAZY (not-yet-computed) xarray
    # objects while the tile loop below builds up the graph. Consumed by dask.compute() once the
    # tile loop finishes, then discarded at the top of the next batch.
    lazy = {name: [] for name in STREAM_NAMES}

    # Iterates through tiles in the batch to create the Dask graph for that batch.
    # That is, the Dask graph/task collection is created tile-by-tile for the batch, rather than for all tiles globally (or for each tile individually). 
    for tile_id in batch_tile_ids:
        subset = features_by_tile[tile_id]
        print(f"  Lazily processing {tile_id}")  # Just constructing the graph for the tile. Not yet doing any processing. 

        tminx, tminy, tmaxx, tmaxy = uu.get_10x10_tile_bounds(tile_id)
        fminx, fminy, fmaxx, fmaxy = subset.total_bounds
        buffer = 0.001

        # Clip the working window to this tile's own features, clamped to the tile itself -- never
        # bigger than the tile, and (after exploding multi-polygon features) never bloated by an unrelated part of
        # the same feature living in a different tile. buffer pads the feature bbox out slightly
        # before clamping, as a safety margin against edge pixels getting cut off at the boundary.
        minx = max(tminx, fminx - buffer)
        miny = max(tminy, fminy - buffer)
        maxx = min(tmaxx, fmaxx + buffer)
        maxy = min(tmaxy, fmaxy + buffer)

        ids = subset[field].astype("uint32").values
        shapes = list(zip(subset.geometry, ids))
        unique_ids = np.unique(ids)

        # Pixel area and optional contextual layer slicing
        pixel_area_bbox = pixel_area.sel(x=slice(minx, maxx), y=slice(maxy, miny))
        pixel_area_ha = pixel_area_bbox * cn.m2_to_ha
        if INCLUDE_LAND_STATE:
            land_state_bbox = land_state_da.sel(x=slice(minx, maxx), y=slice(maxy, miny))
        if INCLUDE_PRIMARY_FOREST:
            primary_forest_bbox = primary_forest_da.sel(x=slice(minx, maxx), y=slice(maxy, miny))

        # Stacks vegetation and LULUCF together along a synthetic "analysis_layer" dimension so both
        # can share one rasterize + one xarray_reduce pass below, rather than needing their own.
        # Only these can be combined because they are the only annual datasets. 
        annual_bboxes = [
            da_.sel(x=slice(minx, maxx), y=slice(maxy, miny)).assign_coords(analysis_layer=label)
            for label, da_ in annual_analysis_layers.items()
        ]
        veg_stacked = xr.concat(annual_bboxes, dim="analysis_layer").rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")
        transform = veg_stacked.rio.transform()

        # The base zone raster for this tile: every feature drawn together with the plain
        # NO_ZONE_ID-filled rasterizer (last-shape-drawn wins on overlapping pixels). Built once here
        # against the veg grid, then relabeled onto each source's own grid via xr.align below.
        zone_id_dask = da.map_blocks(
            rasterize_chunk, veg_stacked.isel(year=0, analysis_layer=0).data,
            shapes=shapes, transform=transform, dtype="uint32",
        )
        zone_da = xr.DataArray(zone_id_dask, dims=("y", "x"),
                               coords={"y": veg_stacked["y"], "x": veg_stacked["x"]}, name=field)

        # Builds the list of arrays to align, conditionally including land_state/primary_forest --
        # the first three positions (veg_stacked, pixel_area_ha, zone_da) are always present, so
        # they can always be unpacked from fixed positions; whatever comes after is popped off in
        # the same order it was appended.
        align_inputs = [veg_stacked, pixel_area_ha, zone_da]
        if INCLUDE_LAND_STATE:
            align_inputs.append(land_state_bbox)
        if INCLUDE_PRIMARY_FOREST:
            align_inputs.append(primary_forest_bbox)
        aligned = xr.align(*align_inputs, join="override")
        annual_aligned, pixel_area_ha_v, zone_da_v = aligned[:3]
        rest = list(aligned[3:])
        if INCLUDE_LAND_STATE:
            land_state_v = rest.pop(0)
        if INCLUDE_PRIMARY_FOREST:
            primary_forest_v = rest.pop(0)

        annual_per_pixel = annual_aligned * pixel_area_ha_v

        # Same idea for the reduce's grouping dimensions: zone is always present; land_state and
        # primary_forest are appended only if enabled, with expected_groups built to match 1:1.
        extra_groupers = []
        extra_expected = []
        if INCLUDE_LAND_STATE:
            extra_groupers.append(land_state_v)
            extra_expected.append(node_codes)
        if INCLUDE_PRIMARY_FOREST:
            extra_groupers.append(primary_forest_v)
            extra_expected.append(cn.composite_primary_codes)

        veg_reduced = xarray_reduce(
            annual_per_pixel, zone_da_v, *extra_groupers, annual_per_pixel["year"],
            func="sum",
            expected_groups=(unique_ids, *extra_expected, annual_per_pixel["year"].values),
            fill_value=0,
        )

        min_soil_bboxes = [
            da_.sel(x=slice(minx, maxx), y=slice(maxy, miny)).assign_coords(analysis_layer=label)
            for label, da_ in min_soil_layers_static.items()
        ]
        min_soil_stacked = xr.concat(min_soil_bboxes, dim="analysis_layer").rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")

        align_inputs = [min_soil_stacked, pixel_area_ha, zone_da]
        if INCLUDE_PRIMARY_FOREST:
            align_inputs.append(primary_forest_bbox)
        aligned = xr.align(*align_inputs, join="override")
        min_soil_aligned, pixel_area_ha_s, zone_da_s = aligned[:3]
        if INCLUDE_PRIMARY_FOREST:
            primary_forest_s = aligned[3]

        min_soil_per_pixel = min_soil_aligned * pixel_area_ha_s

        extra_groupers = []
        extra_expected = []
        if INCLUDE_PRIMARY_FOREST:
            extra_groupers.append(primary_forest_s)
            extra_expected.append(cn.composite_primary_codes)

        min_soil_reduced = xarray_reduce(
            min_soil_per_pixel, zone_da_s, *extra_groupers,
            func="sum", expected_groups=(unique_ids, *extra_expected), fill_value=0,
        )

        organic_bboxes = [
            da_.sel(x=slice(minx, maxx), y=slice(maxy, miny)).assign_coords(analysis_layer=label)
            for label, da_ in organic_soil_layers_native.items()
        ]
        organic_stacked = xr.concat(organic_bboxes, dim="analysis_layer").rio.write_crs("EPSG:4326").rio.set_spatial_dims(x_dim="x", y_dim="y")

        align_inputs = [organic_stacked, pixel_area_ha, zone_da]
        if INCLUDE_PRIMARY_FOREST:
            align_inputs.append(primary_forest_bbox)
        aligned = xr.align(*align_inputs, join="override")
        organic_aligned, pixel_area_ha_o, zone_da_o = aligned[:3]
        if INCLUDE_PRIMARY_FOREST:
            primary_forest_o = aligned[3]

        organic_per_pixel = organic_aligned * pixel_area_ha_o

        extra_groupers = []
        extra_expected = []
        if INCLUDE_PRIMARY_FOREST:
            extra_groupers.append(primary_forest_o)
            extra_expected.append(cn.composite_primary_codes)

        organic_reduced = xarray_reduce(
            organic_per_pixel, zone_da_o, *extra_groupers, organic_per_pixel["year"],
            func="sum",
            expected_groups=(unique_ids, *extra_expected, organic_per_pixel["year"].values),
            fill_value=0,
        )

        # Nothing has actually been computed yet at this point -- these are all still lazy dask/
        # xarray objects. Appending just registers them into this batch's graph; dask.compute() below
        # (after the tile loop) is what actually triggers execution on the cluster.
        lazy["annual"].append((tile_id, veg_reduced))
        lazy["min_soil"].append((tile_id, min_soil_reduced))
        lazy["organic"].append((tile_id, organic_reduced))

        # Total area in raster-land actually captured by rasterizing this tile's zones. Compared later 
        # against feature_total_area_ha (the true vector-geometry area) to flag zones where
        # rasterization lost or gained meaningful area relative to the source shapefile (QC).
        lazy["rasterized_area"].append((tile_id, xarray_reduce(
            pixel_area_ha_v, zone_da_v, func="sum", expected_groups=unique_ids, fill_value=0,
        )))

        # Per-variable NoData footprint (used for flux-per-hectare and the coverage diagnostic further
        # down). Deliberately derived here from arrays already being reduced for the flux
        # totals, rather than via dedicated pixel_count/data_count reduces. An earlier version had 4
        # extra dedicated reduces per tile purely for this diagnostic, and those were the specific
        # root cause of a persistent KilledWorker crash on one particularly heavy tile -- removing
        # them (deriving the same "does this zone have real data" signal post-hoc from
        # rasterized_area_ha==0 / *_data_area_ha==0 instead) fixed the crash outright, with zero loss
        # of diagnostic information.
        annual_has_data = annual_aligned.notnull().any(dim="year")
        min_soil_has_data = min_soil_aligned.notnull()
        organic_has_data = organic_aligned.notnull().any(dim="year")

        lazy["annual_data_area"].append((tile_id, xarray_reduce(
            (annual_has_data.astype("float64") * pixel_area_ha_v), zone_da_v,
            func="sum", expected_groups=unique_ids, fill_value=0,
        )))
        lazy["min_soil_data_area"].append((tile_id, xarray_reduce(
            (min_soil_has_data.astype("float64") * pixel_area_ha_s), zone_da_s,
            func="sum", expected_groups=unique_ids, fill_value=0,
        )))
        lazy["organic_data_area"].append((tile_id, xarray_reduce(
            (organic_has_data.astype("float64") * pixel_area_ha_o), zone_da_o,
            func="sum", expected_groups=unique_ids, fill_value=0,
        )))

        # Overlap-correction pass: one combined rasterize+reduce per layer (see earlier cell for how
        # layers are assigned), each clipped via layer_bounds() to just that layer's own zones'
        # bounding box, not the tile's full window. This per-layer clipping is what fixed an
        # earlier graph-build-time blowup: before it existed, every layer's mask spanned the whole
        # tile regardless of how few zones were actually in it, so a tile needing several correction
        # layers paid for several FULL-TILE-sized passes on top of the base pass.
        for layer_ids in tile_layers.get(tile_id, []):
            layer_id_set = set(layer_ids)
            layer_ids_arr = np.array(layer_ids, dtype="uint32")
            layer_geoms_ids = [(geom, zid) for geom, zid in zip(subset.geometry, ids) if zid in layer_id_set]

            layer_geoms = [g for g, _ in layer_geoms_ids]
            lminx, lminy, lmaxx, lmaxy = layer_bounds(layer_geoms, minx, miny, maxx, maxy, buffer)
            lwin = dict(x=slice(lminx, lmaxx), y=slice(lmaxy, lminy))

            annual_layer_win = annual_per_pixel.sel(**lwin)
            if INCLUDE_LAND_STATE:
                land_state_layer_win = land_state_v.sel(**lwin)
            if INCLUDE_PRIMARY_FOREST:
                primary_forest_v_layer_win = primary_forest_v.sel(**lwin)
                primary_forest_s_layer_win = primary_forest_s.sel(**lwin)
                primary_forest_o_layer_win = primary_forest_o.sel(**lwin)
            min_soil_layer_win = min_soil_per_pixel.sel(**lwin)
            organic_layer_win = organic_per_pixel.sel(**lwin)
            pixel_area_ha_v_layer_win = pixel_area_ha_v.sel(**lwin)
            pixel_area_ha_s_layer_win = pixel_area_ha_s.sel(**lwin)
            pixel_area_ha_o_layer_win = pixel_area_ha_o.sel(**lwin)
            annual_has_data_layer_win = annual_has_data.sel(**lwin)
            min_soil_has_data_layer_win = min_soil_has_data.sel(**lwin)
            organic_has_data_layer_win = organic_has_data.sel(**lwin)

            # Three separate rasterize_layer calls (one per source), matching the three-way
            # xr.align split above for the same reason -- each source's clipped window can select a
            # different pixel count even for "the same" lwin bounds.
            layer_zone_v = rasterize_layer(annual_layer_win.isel(year=0, analysis_layer=0), annual_layer_win.rio.transform(), layer_geoms_ids)
            layer_zone_s = rasterize_layer(min_soil_layer_win.isel(analysis_layer=0), min_soil_layer_win.rio.transform(), layer_geoms_ids)
            layer_zone_o = rasterize_layer(organic_layer_win.isel(year=0, analysis_layer=0), organic_layer_win.rio.transform(), layer_geoms_ids)

            # These seven reduces mirror the base-pass reduces above exactly, but restricted to just
            # this layer's own clipped window and its own zone IDs -- since no two zones sharing a
            # layer overlap each other, these values are correct with no pixel competition. These are
            # the results cell 14 actually keeps for these zones; the base-pass values computed for
            # them above get dropped there instead.
            extra_groupers = []
            extra_expected = []
            if INCLUDE_LAND_STATE:
                extra_groupers.append(land_state_layer_win)
                extra_expected.append(node_codes)
            if INCLUDE_PRIMARY_FOREST:
                extra_groupers.append(primary_forest_v_layer_win)
                extra_expected.append(cn.composite_primary_codes)
            lazy["overlap_annual"].append((tile_id, xarray_reduce(
                annual_layer_win, layer_zone_v, *extra_groupers, annual_layer_win["year"],
                func="sum",
                expected_groups=(layer_ids_arr, *extra_expected, annual_layer_win["year"].values),
                fill_value=0,
            )))
            
            extra_groupers = []
            extra_expected = []
            if INCLUDE_PRIMARY_FOREST:
                extra_groupers.append(primary_forest_s_layer_win)
                extra_expected.append(cn.composite_primary_codes)

            lazy["overlap_min_soil"].append((tile_id, xarray_reduce(
                min_soil_layer_win, layer_zone_s, *extra_groupers,
                func="sum", expected_groups=(layer_ids_arr, *extra_expected), fill_value=0,
            )))

            extra_groupers = []
            extra_expected = []
            if INCLUDE_PRIMARY_FOREST:
                extra_groupers.append(primary_forest_o_layer_win)
                extra_expected.append(cn.composite_primary_codes)
            lazy["overlap_organic"].append((tile_id, xarray_reduce(
                organic_layer_win, layer_zone_o, *extra_groupers, organic_layer_win["year"],
                func="sum",
                expected_groups=(layer_ids_arr, *extra_expected, organic_layer_win["year"].values),
                fill_value=0,
            )))
                
            lazy["overlap_area"].append((tile_id, xarray_reduce(
                pixel_area_ha_v_layer_win, layer_zone_v, func="sum", expected_groups=layer_ids_arr, fill_value=0,
            )))
            lazy["overlap_annual_data_area"].append((tile_id, xarray_reduce(
                (annual_has_data_layer_win.astype("float64") * pixel_area_ha_v_layer_win), layer_zone_v,
                func="sum", expected_groups=layer_ids_arr, fill_value=0,
            )))
            lazy["overlap_min_soil_data_area"].append((tile_id, xarray_reduce(
                (min_soil_has_data_layer_win.astype("float64") * pixel_area_ha_s_layer_win), layer_zone_s,
                func="sum", expected_groups=layer_ids_arr, fill_value=0,
            )))
            lazy["overlap_organic_data_area"].append((tile_id, xarray_reduce(
                (organic_has_data_layer_win.astype("float64") * pixel_area_ha_o_layer_win), layer_zone_o,
                func="sum", expected_groups=layer_ids_arr, fill_value=0,
            )))

    # Preparing to run on batch graph
    flat_lazy_results = [r for name in STREAM_NAMES for _, r in lazy[name]]
    # collections_to_dsk materializes the full task graph just to count it
    batch_graph = dask.base.collections_to_dsk(flat_lazy_results)
    print(f"  Batch {batch_num + 1} graph: {len(batch_graph)} tasks")

    print(f"  Batch {batch_num + 1} graph built: {uu.timestr()}; computing...")
    compute_start = time.time()
    # Retries the whole batch's compute up to MAX_COMPUTE_ATTEMPTS times if dask.compute() itself
    # raises (e.g. KilledWorker from a worker hitting its memory budget) -- these have shown up as
    # transient, not reliably reproducible on a given batch, so retrying from scratch after a short
    # pause (giving Coiled a chance to replace the lost worker) is a reasonable way to absorb an
    # occasional crash without losing the whole multi-hour run. Prints a WARNING on every failed
    # attempt and a confirmation line if a retry subsequently succeeds, so retries are always visible
    # in the output rather than silent.
    MAX_COMPUTE_ATTEMPTS = 3
    for attempt in range(1, MAX_COMPUTE_ATTEMPTS + 1):
        try:
            with get_task_stream(client=client) as ts:
                computed = dask.compute(*flat_lazy_results)
            if attempt > 1:
                print(f"  Batch {batch_num + 1} succeeded on attempt {attempt}/{MAX_COMPUTE_ATTEMPTS} after retrying.")
            break
        except Exception as e:
            if attempt == MAX_COMPUTE_ATTEMPTS:
                raise
            print(f"  WARNING: batch {batch_num + 1} attempt {attempt}/{MAX_COMPUTE_ATTEMPTS} failed "
                  f"({type(e).__name__}: {e}) -- retrying the whole batch after a short pause...")
            time.sleep(30)
    compute_duration = time.time() - compute_start
    total_compute_duration_sec += compute_duration
    print(f"  Batch {batch_num + 1} compute time: {compute_duration/60:.1f} min ({compute_duration:.0f} sec)")

    total_expected_tasks += len(batch_graph)
    total_actual_tasks += len(ts.data)
    print(f"  Batch {batch_num + 1}: {len(batch_graph)} expected tasks, {len(ts.data)} actually ran; difference of {len(ts.data)-len(batch_graph)}")

    # Reassembles results from all the streams that were merged together during computation into separate results
    idx = 0
    for name in STREAM_NAMES:
        n = len(lazy[name])
        batch_computed = computed[idx:idx+n]
        results[name].extend((t, r) for (t, _), r in zip(lazy[name], batch_computed))
        idx += n

        # Per-batch CSV checkpoint -- written immediately after each batch computes, not just held
        # in memory in results[name]. This is what makes it possible to resume a long run from the
        # last completed batch instead of losing everything if a later batch crashes.
        if lazy[name]:
            dfs = []
            for (t, _), r in zip(lazy[name], batch_computed):
                df = r.to_dataframe(name="value").reset_index()
                df["tile_id"] = t
                dfs.append(df)
            batch_df = pd.concat(dfs, ignore_index=True)
            batch_df.to_csv(f"{run_folder}/batch_{batch_num + 1:02d}_{name}.csv", index=False)

    # Total wall-clock for this batch (graph build + compute + CSV writes) -- compute_duration above
    # is just the dask.compute() portion, so the gap between the two shows local overhead.
    batch_duration = time.time() - batch_start
    total_duration_sec += batch_duration
    print(f"  Batch {batch_num + 1} duration: {batch_duration/60:.1f} min ({batch_duration:.0f} sec)")

print(f"\nAll batches complete: {uu.timestr()}")
print(f"Total across all batches: {total_expected_tasks} expected tasks, {total_actual_tasks} actually ran")
print(f"Total compute time across all batches: {total_compute_duration_sec/60:.1f} min ({total_compute_duration_sec:.0f} sec)")
print(f"Total duration across all batches: {total_duration_sec/60:.1f} min ({total_duration_sec:.0f} sec)")
cluster.shutdown()  # explicit teardown rather than relying on idle_timeout, so idle cluster time
                     # after the loop finishes isn't billed unnecessarily

Saving batch outputs to: /mnt/c/GIS/AFOLU_flux_model/shapefile_zonal_stats/Brazil_biomes_20261014_141803

=== Batch 1/2: 8 tiles -- building graph: 20261014_14_19_50 ===
  Lazily processing 10S_070W
  Lazily processing 20S_060W
  Lazily processing 00N_060W
  Lazily processing 10S_050W
  Lazily processing 10N_070W
  Lazily processing 10S_040W
  Lazily processing 00N_080W
  Lazily processing 10S_080W
  Batch 1 graph: 65238 tasks
  Batch 1 graph built: 20261014_14_19_58; computing...


/home/dagibbs22/miniforge3/envs/coiled_20260815/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 16.90 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


  Batch 1 compute time: 5.2 min (314 sec)
  Batch 1: 65238 expected tasks, 69000 actually ran; difference of 3762
  Batch 1 duration: 5.5 min (329 sec)

=== Batch 2/2: 9 tiles -- building graph: 20261014_14_25_19 ===
  Lazily processing 00N_070W
  Lazily processing 10S_060W
  Lazily processing 20S_050W
  Lazily processing 00N_050W
  Lazily processing 10N_060W
  Lazily processing 00N_040W
  Lazily processing 10N_050W
  Lazily processing 30S_060W
  Lazily processing 10N_080W
  Batch 2 graph: 67214 tasks
  Batch 2 graph built: 20261014_14_25_28; computing...


/home/dagibbs22/miniforge3/envs/coiled_20260815/lib/python3.12/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 17.42 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


  Batch 2 compute time: 3.3 min (195 sec)
  Batch 2: 67214 expected tasks, 67214 actually ran; difference of 0
  Batch 2 duration: 3.5 min (211 sec)

All batches complete: 20261014_14_28_50
Total across all batches: 132452 expected tasks, 136214 actually ran
Total compute time across all batches: 8.5 min (509 sec)
Total duration across all batches: 9.0 min (540 sec)
CPU times: user 32.8 s, sys: 983 ms, total: 33.8 s
Wall time: 9min


In [15]:
%%time

### Main table assembly
# Takes every batch's accumulated results (results dict, built up in .compute() cell) and assembles them
# into the single final wide-format table: one row per zone (and per year, for veg/organic), with
# shapefile attributes, area, and overlap metadata merged in.
# INCLUDE_LAND_STATE / INCLUDE_PRIMARY_FOREST (cell 1) control whether this cell's grouping/pivoting
# includes those two contextual dimensions at all -- when a flag is off, the corresponding column
# never existed in the upstream reduce (cell 13), so every list below that used to hardcode those
# column names is now built dynamically to match whatever's actually present.

# General-purpose helper: takes a list of (tile_id, xarray result) pairs (as stored in results[...]),
# flattens every tile's result into one dataframe, and collapses down to one row per zone by summing
# value_name across everything. Used below only for the single-dimension area streams -- the veg/soil
# flux tables need extra dimensions (year, land_state, ...) preserved rather than collapsed, so they
# don't go through this helper.
def sum_by_zone(result_pairs, value_name):
    dfs = [r.to_dataframe(name=value_name).reset_index() for _, r in result_pairs]
    return pd.concat(dfs, ignore_index=True).groupby(field, as_index=False)[value_name].sum()

print("Creating table with annual datasets (vegetation and LULUCF)")
veg_dfs = []
for tile_id, result in results["annual"]:
    df = result.to_dataframe(name="value").reset_index()
    # Translates the zarr's integer year INDEX (0-8, what xarray_reduce actually grouped by) back
    # into the real calendar year (2016-2024) for the output table.
    df["year"] = df["year"].map(lambda i: years[i])
    veg_dfs.append(df)
veg_combined = pd.concat(veg_dfs, ignore_index=True)
# Drops every row for a zone that needed overlap correction -- these base-pass values lost pixels in
# the rasterize tie-break and are wrong. The correct values get spliced in from results["overlap_annual"]
# next.
veg_combined = veg_combined[~veg_combined[field].isin(overlap_needs_correction_ids)]

overlap_veg_dfs = []
for tile_id, result in results["overlap_annual"]:
    df = result.to_dataframe(name="value").reset_index()
    df["year"] = df["year"].map(lambda i: years[i])
    overlap_veg_dfs.append(df)
if overlap_veg_dfs:
    # This drop-then-splice pattern only works because the layering rework (cell 7) gives
    # correction-pass results the exact same shape/schema as the base-pass results -- an earlier
    # design (before layering existed) needed a separate, differently-shaped "overlap table" merged
    # in afterward instead. The `if` guards against concatenating an empty list when nothing in this
    # run needed correcting at all.
    veg_combined = pd.concat([veg_combined] + overlap_veg_dfs, ignore_index=True)

# Grouping columns built dynamically -- land_state_node/the primary_forest column simply aren't
# columns in veg_combined at all when their flag is off (cell 13's reduce never grouped by them), so
# this list has to match whatever's actually present rather than assuming both are there.
veg_group_cols = [field, "year"]
if INCLUDE_LAND_STATE:
    veg_group_cols.append("land_state_node")
if INCLUDE_PRIMARY_FOREST:
    veg_group_cols.append(cn.starting_composite_primary_forest_pattern)
veg_group_cols.append("analysis_layer")

# Re-aggregates after the drop+splice -- a zone can contribute rows from multiple tiles (a feature
# spanning a tile boundary) or, post-splice, from a correction layer, so this collapses everything
# down to one row per zone x year x (whichever contextual columns are active) x analysis_layer.
veg_combined = veg_combined.groupby(veg_group_cols, as_index=False)["value"].sum()
# expected_groups (cell 13) always produces a row for every possible category of whichever
# contextual columns are active, most of which are legitimately zero for any given zone -- this
# drops those empty rows rather than carrying them through to the final table.
veg_combined = veg_combined[veg_combined["value"] != 0]

# These two checks now do double duty: they were originally just defensive (harmless if a column
# somehow wasn't present), but they're also exactly what's needed to make the optional-layers
# feature work -- when a flag is off, its column genuinely isn't in veg_combined, and both merges
# below quietly no-op instead of needing their own separate INCLUDE_* guard.
if "land_state_node" in veg_combined.columns:
    # Attaches the human-readable land-state names/classes (built in cell 10) onto the raw numeric
    # land_state_node codes.
    veg_combined = veg_combined.merge(
        state_node_df[['land_state', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type', 'tree_focused_2x2']],
        left_on="land_state_node", right_on="land_state", how="left",
    ).drop(columns=["land_state"])

if cn.starting_composite_primary_forest_pattern in veg_combined.columns:
    # Converts the raw 0/1 primary-forest code into its "not primary forest"/"primary forest" label.
    veg_combined[cn.starting_composite_primary_forest_pattern] = veg_combined[cn.starting_composite_primary_forest_pattern].map(primary_forest_labels)

display(veg_combined)  # long-format sanity check before pivoting wide below

# Pivot index built the same way as the groupby above -- land_state's extra descriptive columns
# (land_state_meaning, etc., added by the merge just above) only get included when the flag is on,
# since they wouldn't exist otherwise.
veg_index_cols = [field, "year"]
if INCLUDE_LAND_STATE:
    veg_index_cols += ["land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "tall_veg_type", "tree_focused_2x2"]
if INCLUDE_PRIMARY_FOREST:
    veg_index_cols.append(cn.starting_composite_primary_forest_pattern)

veg_wide = veg_combined.pivot_table(
    index=veg_index_cols,
    columns="analysis_layer", values="value", fill_value=0,
).reset_index()
# Pivots from long format (one row per zone/year/.../analysis_layer, one value column) to wide
# format (one row per zone/year/..., one column per analysis_layer: veg_gross_emissions_MgCO2e,
# LULUCF_net_flux_MgCO2e, etc.). These two loops pre-create empty (all-zero) placeholder columns for
# the soil flux variables, so veg_wide ends up with the exact same column set as soil_wide below,
# before the two get stacked together.
for col in min_soil_var_names:
    veg_wide[col] = 0
for col in organic_soil_var_names:
    veg_wide[col] = 0

print("Creating mineral soil table")
# Same base-pass building as veg, but mineral soil has no year dimension to translate -- it's a
# single static value per zone (see cell 9), so there's no equivalent of the years[i] mapping step.
min_soil_dfs = [result.to_dataframe(name="value").reset_index() for _, result in results["min_soil"]]
min_soil_combined = pd.concat(min_soil_dfs, ignore_index=True)
min_soil_combined = min_soil_combined[~min_soil_combined[field].isin(overlap_needs_correction_ids)]

overlap_min_soil_dfs = [result.to_dataframe(name="value").reset_index() for _, result in results["overlap_min_soil"]]
if overlap_min_soil_dfs:
    min_soil_combined = pd.concat([min_soil_combined] + overlap_min_soil_dfs, ignore_index=True)

# Mineral soil never had a land_state column to begin with -- only primary_forest is conditional here.
min_soil_group_cols = [field]
if INCLUDE_PRIMARY_FOREST:
    min_soil_group_cols.append(cn.starting_composite_primary_forest_pattern)
min_soil_group_cols.append("analysis_layer")

min_soil_combined = min_soil_combined.groupby(min_soil_group_cols, as_index=False)["value"].sum()
if INCLUDE_PRIMARY_FOREST:
    min_soil_combined[cn.starting_composite_primary_forest_pattern] = min_soil_combined[cn.starting_composite_primary_forest_pattern].map(primary_forest_labels)

min_soil_index_cols = [field] + ([cn.starting_composite_primary_forest_pattern] if INCLUDE_PRIMARY_FOREST else [])
min_soil_wide = min_soil_combined.pivot_table(
    index=min_soil_index_cols, columns="analysis_layer", values="value", fill_value=0,
).reset_index()
# Mineral soil's one static value gets broadcast across every analysis year via a cross join (every
# row paired with every year in `years`) -- deliberately repeating the same number for every year,
# not because the soil actually changed, but so the final table has one row per zone per year,
# matching the shape of the veg/organic tables it's combined with below.
min_soil_wide = min_soil_wide.merge(pd.DataFrame({"year": years}), how="cross")

print("Creating organic soil table")
organic_dfs = [result.to_dataframe(name="value").reset_index() for _, result in results["organic"]]
organic_combined = pd.concat(organic_dfs, ignore_index=True)
organic_combined = organic_combined[~organic_combined[field].isin(overlap_needs_correction_ids)]

overlap_organic_dfs = [result.to_dataframe(name="value").reset_index() for _, result in results["overlap_organic"]]
if overlap_organic_dfs:
    organic_combined = pd.concat([organic_combined] + overlap_organic_dfs, ignore_index=True)

# Organic soil never had a land_state column either -- same treatment as mineral soil above.
organic_group_cols = [field, "year"]
if INCLUDE_PRIMARY_FOREST:
    organic_group_cols.append(cn.starting_composite_primary_forest_pattern)
organic_group_cols.append("analysis_layer")

organic_combined = organic_combined.groupby(organic_group_cols, as_index=False)["value"].sum()
if INCLUDE_PRIMARY_FOREST:
    organic_combined[cn.starting_composite_primary_forest_pattern] = organic_combined[cn.starting_composite_primary_forest_pattern].map(primary_forest_labels)
# "year" here is still the organic-soil INTERVAL index (0 or 1, for the two 5-year blocks), not yet
# a real calendar year -- that gets resolved next.
organic_index_cols = [field, "year"] + ([cn.starting_composite_primary_forest_pattern] if INCLUDE_PRIMARY_FOREST else [])
organic_wide = organic_combined.pivot_table(
    index=organic_index_cols, columns="analysis_layer", values="value", fill_value=0,
).reset_index()

def organic_soil_interval_idx(year):
    for i, interval in enumerate(cn.organic_soil_year_intervals):
        start, end = (int(y) for y in interval.split("_"))
        if start <= year <= end:
            return i
    raise ValueError(f"No organic soil interval covers year {year}")

# Maps each real analysis year onto which 5-year interval covers it, then merges that mapping in to
# expand organic_wide from "one row per zone per interval" to "one row per zone per real year" --
# broadcasting (repeating) the same interval value across every year within it, deliberately, not
# interpolating. This is what makes organic soil's shape finally match veg's (one row per zone per
# real year) so the two can be combined below.
year_to_interval_idx = pd.DataFrame({"year": years, "interval_idx": [organic_soil_interval_idx(y) for y in years]})
organic_wide = organic_wide.rename(columns={"year": "interval_idx"}).merge(
    year_to_interval_idx, on="interval_idx", how="left"
).drop(columns="interval_idx")

print("Combining mineral and organic soil tables")
# The merge key mirrors the group/pivot columns above -- primary_forest only joins on if it's
# actually a shared column between the two tables.
soil_merge_cols = [field, "year"] + ([cn.starting_composite_primary_forest_pattern] if INCLUDE_PRIMARY_FOREST else [])
soil_wide = min_soil_wide.merge(organic_wide, on=soil_merge_cols, how="outer")
# Soil rows have no real land-state/vegetation-type concept (that's a vegetation-specific
# classification) -- tagged with clearly-synthetic placeholder values so a soil row is never mistaken
# for a genuine vegetation land-state category downstream or in the final output. Only added when
# INCLUDE_LAND_STATE is on -- otherwise veg_wide has no land_state columns either, and adding them
# only here would leave soil_wide with columns veg_wide doesn't share.
if INCLUDE_LAND_STATE:
    soil_wide["land_state_node"] = 0
    soil_wide["land_state_meaning"] = "no_land_state__soil"
    soil_wide["land_state_broad_class"] = "no_land_state__soil"
    soil_wide["land_state_detailed_class"] = "no_land_state__soil"
    soil_wide["tall_veg_type"] = "no_land_state__soil"
    soil_wide["tree_focused_2x2"] = "no_land_state__soil"
# Mirrors what veg_wide did for the soil columns above -- adds empty (all-zero) veg/LULUCF flux
# columns here, so both tables have an identical column set before being stacked together next.
for col in list(veg_var_names.keys()) + list(lulucf_var_names.keys()):
    soil_wide[col] = 0

print("Combining tables")
# Stacks veg/LULUCF rows and soil rows into one table. Because both were engineered above to share
# an identical column set, this is a plain vertical concatenation -- but note it produces TWO
# distinct kinds of row per zone/year, not one row holding both veg and soil together: a veg row
# (real veg/LULUCF flux, soil columns = 0, broken out by whichever contextual columns are active)
# and a soil row (real soil flux, veg/LULUCF columns = 0, tagged with the placeholder land-state
# values above when land_state is active). They're kept separate rather than merged into one because
# veg data is naturally split across multiple contextual categories per zone per year, while soil is
# a single number per zone per year -- forcing them into one row per zone/year would require an
# arbitrary choice of which category the soil value "belongs" to.
wide_combined = pd.concat([veg_wide, soil_wide], ignore_index=True)

# Same drop-then-splice pattern as the flux tables above, generalized as a reusable helper for
# area-only streams. These have no year/land-state dimension to preserve (just one area-per-zone
# total), so sum_by_zone's simple collapse-to-one-row-per-zone is exactly what's needed here, unlike
# the flux tables which needed groupby to keep extra dimensions. Unaffected by the contextual-layer
# flags -- these streams were never grouped by land_state/primary_forest in cell 13.
def area_with_correction(base_results, overlap_results, value_name):
    base = sum_by_zone(base_results, value_name)
    base = base[~base[field].isin(overlap_needs_correction_ids)]
    if overlap_results:
        corrected = sum_by_zone(overlap_results, value_name)
        base = pd.concat([base, corrected], ignore_index=True)
    return base

# Like sum_by_zone, but preserves the analysis_layer dimension -- needed for the *_data_area_ha
# streams, since each source variable (e.g. veg_gross_emissions vs veg_net_flux) can have slightly
# different data coverage and needs its own area total, not one collapsed number per zone.
def area_by_zone_and_layer(result_pairs):
    dfs = [r.to_dataframe(name="area_ha").reset_index() for _, r in result_pairs]
    return pd.concat(dfs, ignore_index=True).groupby([field, "analysis_layer"], as_index=False)["area_ha"].sum()

# Same drop-then-splice as area_with_correction, but for the per-analysis-layer area streams --
# pivots wide (one column per analysis_layer) and renames each column with a _data_area_ha suffix
# (e.g. veg_gross_emissions_MgCO2e_data_area_ha) so it's clearly distinguishable from the flux column
# of the same base name once merged into the final table.
def area_with_correction_by_layer(base_results, overlap_results):
    base = area_by_zone_and_layer(base_results)
    base = base[~base[field].isin(overlap_needs_correction_ids)]
    if overlap_results:
        corrected = area_by_zone_and_layer(overlap_results)
        base = pd.concat([base, corrected], ignore_index=True)
    wide = base.pivot_table(index=field, columns="analysis_layer", values="area_ha", fill_value=0).reset_index()
    return wide.rename(columns={c: f"{c}_data_area_ha" for c in wide.columns if c != field})

# Builds one combined per-zone area table: starts with total rasterized_area_ha (one number per
# zone), then merges in each source's per-analysis-layer data-coverage area (one set of columns per
# source), joining everything by zone.
area_totals = area_with_correction(results["rasterized_area"], results["overlap_area"], "rasterized_area_ha")
for base_r, overlap_r in [
    (results["annual_data_area"], results["overlap_annual_data_area"]),
    (results["min_soil_data_area"], results["overlap_min_soil_data_area"]),
    (results["organic_data_area"], results["overlap_organic_data_area"]),
]:
    area_totals = area_totals.merge(area_with_correction_by_layer(base_r, overlap_r), on=field, how="outer")

# drop_duplicates(subset=field): after exploding (cell 6), gdf has multiple rows per zone (one per
# part) -- without this, the merge would silently become many-to-one and duplicate every veg/soil
# row for that zone once per part it has. .astype({field: "uint32"}) is a defensive cast, redundant
# here since raster_id is already uint32 from creation, kept for consistency with versions of this
# notebook where field isn't guaranteed to already be that type.
wide_combined = wide_combined.merge(
    gdf.drop_duplicates(subset=field)[[field] + output_fields + ["feature_total_area_ha"]].astype({field: "uint32"}), on=field, how="left"
)
wide_combined = wide_combined.merge(area_totals, on=field, how="left")

# QC column: how much the rasterized area differs from the true vector area for each zone -- a large
# gap can flag rasterization issues (very small/sliver features, antimeridian-straddling geometries).
wide_combined["area_diff_ha"] = wide_combined["feature_total_area_ha"] - wide_combined["rasterized_area_ha"]

def format_overlap_ids(zone_id):
    partners = overlap_partner_ids.get(zone_id)
    return ";".join(str(p) for p in sorted(partners)) if partners else ""

# Human-readable, semicolon-joined list of every zone this one overlaps with (from cell 7's
# adjacency map), or an empty string if it has none -- purely informational, for auditing which
# totals were affected by overlap correction.
wide_combined["overlapping_ids"] = wide_combined[field].map(format_overlap_ids)

# With by-GHG layers, rebuild the standard fire/drainage subtotals and the veg gross-emissions total
# so the reconciliation tab and comparisons to earlier runs keep working.
if USE_GHG_SPECIFIC_EMISSIONS:
    wide_combined["organic_soil_emissions_fire_MgCO2e"] = wide_combined[organic_burned_labels].sum(axis=1)
    wide_combined["organic_soil_emissions_drainage_MgCO2e"] = wide_combined[organic_drained_labels].sum(axis=1)
    wide_combined["veg_gross_emissions_MgCO2e"] = wide_combined[veg_gross_emission_labels].sum(axis=1)

# Derived total combining organic soil's two separate emission sources -- also feeds the
# reconciliation tab (cell 16).
wide_combined["organic_soil_emissions_total_MgCO2e"] = wide_combined["organic_soil_emissions_fire_MgCO2e"] + wide_combined["organic_soil_emissions_drainage_MgCO2e"]

# Groups column names by category (LULUCF/veg/soil) purely to control the final column ordering
# below -- doesn't touch any data.
flux_cols_lulucf = list(lulucf_var_names.keys())
flux_cols_veg = list(veg_var_names.keys())
flux_cols_soil = list(min_soil_var_names.keys()) + list(organic_soil_var_names.keys()) + ["organic_soil_emissions_total_MgCO2e"]
if USE_GHG_SPECIFIC_EMISSIONS:
    flux_cols_veg = ["veg_gross_emissions_MgCO2e"] + flux_cols_veg
    flux_cols_soil = flux_cols_soil + ["organic_soil_emissions_fire_MgCO2e", "organic_soil_emissions_drainage_MgCO2e"]

area_data_cols_lulucf = [c for c in wide_combined.columns if c.endswith("_data_area_ha") and c.startswith("LULUCF_")]
area_data_cols_veg = [c for c in wide_combined.columns if c.endswith("_data_area_ha") and c.startswith("veg_")]
area_data_cols_soil = [c for c in wide_combined.columns if c.endswith("_data_area_ha") and (c.startswith("mineral_soil_") or c.startswith("organic_soil_"))]

# Contextual column block built dynamically -- land_state's descriptive columns and/or the
# primary_forest column simply aren't in wide_combined at all when their flag was off, so hardcoding
# this list (as before) would raise a KeyError on the final selection below.
contextual_cols = ["year"]
if INCLUDE_LAND_STATE:
    contextual_cols += ["land_state_node", "land_state_meaning", "land_state_broad_class", "land_state_detailed_class", "tall_veg_type", "tree_focused_2x2"]
if INCLUDE_PRIMARY_FOREST:
    contextual_cols.append(cn.starting_composite_primary_forest_pattern)

# Final column order: shapefile identity/attributes -> area/overlap QC columns -> contextual
# (land-state/primary-forest, whichever are active) columns -> flux columns grouped LULUCF/veg/soil
# -> their corresponding area-coverage columns in the same grouping. Purely cosmetic, doesn't change
# any values.
wide_combined = wide_combined[
    [field] + output_fields
    + ["feature_total_area_ha", "rasterized_area_ha", "area_diff_ha", "overlapping_ids"]
    + contextual_cols
    + flux_cols_lulucf + flux_cols_veg + flux_cols_soil
    + area_data_cols_lulucf + area_data_cols_veg + area_data_cols_soil
]

print(f"Rows in dataframe: {len(wide_combined)}: {uu.timestr()}")
display(wide_combined)

Creating table with annual datasets (vegetation and LULUCF)


,raster_id,year,land_state_node,starting_composite_primary_forest,analysis_layer,value,land_state_meaning,land_state_broad_class,land_state_detailed_class,tall_veg_type,tree_focused_2x2
0,0,2016,10000000,not primary forest,LULUCF_gross_emissions_MgCO2e,2.710730e+02,Before mangrove gain (mangrove mask),tree,mangrove_other,mangrove,tree_gain
1,0,2016,10000000,not primary forest,LULUCF_gross_removals_MgCO2e,-2.304474e+02,Before mangrove gain (mangrove mask),tree,mangrove_other,mangrove,tree_gain
2,0,2016,10000000,not primary forest,LULUCF_net_flux_MgCO2e,4.062549e+01,Before mangrove gain (mangrove mask),tree,mangrove_other,mangrove,tree_gain
3,0,2016,10000000,primary forest,LULUCF_gross_emissions_MgCO2e,6.042275e+01,Before mangrove gain (mangrove mask),tree,mangrove_other,mangrove,tree_gain
4,0,2016,10000000,primary forest,LULUCF_gross_removals_MgCO2e,-5.523512e+01,Before mangrove gain (mangrove mask),tree,mangrove_other,mangrove,tree_gain
...,...,...,...,...,...,...,...,...,...,...,...
28894,5,2024,70000000,not primary forest,LULUCF_gross_removals_MgCO2e,-2.732593e+06,Non-vegetated land remaining non-vegetated land,no_flux,no_flux,non_tall_vegetation,non_trees_remaining_non_trees
28895,5,2024,70000000,not primary forest,LULUCF_net_flux_MgCO2e,8.735683e+06,Non-vegetated land remaining non-vegetated land,no_flux,no_flux,non_tall_vegetation,non_trees_remaining_non_trees
28896,5,2024,70000000,primary forest,LULUCF_gross_emissions_MgCO2e,4.903855e+04,Non-vegetated land remaining non-vegetated land,no_flux,no_flux,non_tall_vegetation,non_trees_remaining_non_trees
28897,5,2024,70000000,primary forest,LULUCF_gross_removals_MgCO2e,-1.414904e+04,Non-vegetated land remaining non-vegetated land,no_flux,no_flux,non_tall_vegetation,non_trees_remaining_non_trees


Creating mineral soil table
Creating organic soil table
Combining mineral and organic soil tables
Combining tables
Rows in dataframe: 5737: 20261014_14_29_04


,raster_id,Name,feature_total_area_ha,rasterized_area_ha,area_diff_ha,overlapping_ids,year,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,tall_veg_type,tree_focused_2x2,starting_composite_primary_forest,LULUCF_gross_emissions_MgCO2e,LULUCF_gross_removals_MgCO2e,LULUCF_net_flux_MgCO2e,veg_gross_emissions_MgCO2e,veg_gross_removals_MgCO2e,veg_net_flux_MgCO2e,mineral_soil_gross_loss_MgCO2,mineral_soil_gross_gain_MgCO2,mineral_soil_net_change_MgCO2,organic_soil_emissions_fire_MgCO2e,organic_soil_emissions_drainage_MgCO2e,organic_soil_emissions_total_MgCO2e,LULUCF_gross_emissions_MgCO2e_data_area_ha,LULUCF_gross_removals_MgCO2e_data_area_ha,LULUCF_net_flux_MgCO2e_data_area_ha,veg_gross_emissions_MgCO2e_data_area_ha,veg_gross_removals_MgCO2e_data_area_ha,veg_net_flux_MgCO2e_data_area_ha,mineral_soil_gross_gain_MgCO2_data_area_ha,mineral_soil_gross_loss_MgCO2_data_area_ha,mineral_soil_net_change_MgCO2_data_area_ha,organic_soil_emissions_drainage_MgCO2e_data_area_ha,organic_soil_emissions_fire_MgCO2e_data_area_ha
0,0,Caatinga,8.265254e+07,82652504.0,40.122933,,2016,10000000,Before mangrove gain (mangrove mask),tree,mangrove_other,mangrove,tree_gain,not primary forest,271.072968,-230.447449,40.625492,0.0,0.0,0.0,0.0,0.00,0.0,0.00000,0.000000,0.000000,3.378919e+07,6.403949e+07,7.857458e+07,8.265250e+07,8.265250e+07,8.265250e+07,5.249215e+07,3.887770e+07,8.210560e+07,8.265250e+07,8.265250e+07
1,0,Caatinga,8.265254e+07,82652504.0,40.122933,,2016,10000000,Before mangrove gain (mangrove mask),tree,mangrove_other,mangrove,tree_gain,primary forest,60.422752,-55.235115,5.187637,0.0,0.0,0.0,0.0,0.00,0.0,0.00000,0.000000,0.000000,3.378919e+07,6.403949e+07,7.857458e+07,8.265250e+07,8.265250e+07,8.265250e+07,5.249215e+07,3.887770e+07,8.210560e+07,8.265250e+07,8.265250e+07
2,0,Caatinga,8.265254e+07,82652504.0,40.122933,,2016,10100000,Non-mangrove remaining non-mangrove (mangrove ...,tree,mangrove_other,mangrove,non_trees_remaining_non_trees,not primary forest,417.011200,-263.019440,153.991776,0.0,0.0,0.0,0.0,0.00,0.0,0.00000,0.000000,0.000000,3.378919e+07,6.403949e+07,7.857458e+07,8.265250e+07,8.265250e+07,8.265250e+07,5.249215e+07,3.887770e+07,8.210560e+07,8.265250e+07,8.265250e+07
3,0,Caatinga,8.265254e+07,82652504.0,40.122933,,2016,10100000,Non-mangrove remaining non-mangrove (mangrove ...,tree,mangrove_other,mangrove,non_trees_remaining_non_trees,primary forest,156.820068,-86.066360,70.753700,0.0,0.0,0.0,0.0,0.00,0.0,0.00000,0.000000,0.000000,3.378919e+07,6.403949e+07,7.857458e+07,8.265250e+07,8.265250e+07,8.265250e+07,5.249215e+07,3.887770e+07,8.210560e+07,8.265250e+07,8.265250e+07
4,0,Caatinga,8.265254e+07,82652504.0,40.122933,,2016,10200000,After permanent mangrove loss (mangrove mask),tree,mangrove_other,mangrove,tree_loss,not primary forest,2040.266968,-354.781281,1685.485840,0.0,0.0,0.0,0.0,0.00,0.0,0.00000,0.000000,0.000000,3.378919e+07,6.403949e+07,7.857458e+07,8.265250e+07,8.265250e+07,8.265250e+07,5.249215e+07,3.887770e+07,8.210560e+07,8.265250e+07,8.265250e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5732,5,Mata Atlântica,1.106136e+08,110613440.0,182.196617,,2022,0,no_land_state__soil,no_land_state__soil,no_land_state__soil,no_land_state__soil,no_land_state__soil,primary forest,0.000000,0.000000,0.000000,0.0,0.0,0.0,7630137.5,-1485743.25,6144395.0,0.00000,23.961939,23.961939,7.066895e+07,7.454334e+07,1.061248e+08,1.106134e+08,1.106134e+08,1.106134e+08,4.392360e+07,7.149313e+07,1.088631e+08,1.106134e+08,1.106134e+08
5733,5,Mata Atlântica,1.106136e+08,110613440.0,182.196617,,2023,0,no_land_state__soil,no_land_state__soil,no_land_state__soil,no_land_state__soil,no_land_state__soil,not primary forest,0.000000,0.000000,0.000000,0.0,0.0,0.0,100554920.0,-42839508.00,57715416.0,81.31633,1897.899536,1979.215866,7.066895e+07,7.454334e+07,1.061248e+08,1.106134e+08,1.106134e+08,1.106134e+08,4.392360e+

CPU times: user 1.85 s, sys: 88.5 ms, total: 1.94 s
Wall time: 13 s


In [16]:
### Diagnostic coverage

# Runs AFTER assembly because it reads wide_combined.
# this cell was moved down when it was reconceived to derive its
# answer post-hoc from analysis layer area data that's already being computed (rasterized_area_ha, *_data_area_ha),
# rather than from 4 dedicated pixel_count/data_count reduces added per tile in the batch loop. That
# redesign fixed a persistent KilledWorker crash on one particular tile with zero loss of diagnostic
# value: rasterized_area_ha==0 is exactly equivalent to the old pixel_count==0, and every relevant
# *_data_area_ha==0 for a source is exactly equivalent to the old data_pixel_count==0.

data_area_cols = {
    "vegetation_LULUCF": [c for c in wide_combined.columns if c.endswith("_data_area_ha") and (c.startswith("veg_") or c.startswith("LULUCF_"))],
    "mineral_soil": [c for c in wide_combined.columns if c.endswith("_data_area_ha") and c.startswith("mineral_soil_")],
    "organic_soil": [c for c in wide_combined.columns if c.endswith("_data_area_ha") and c.startswith("organic_soil_")],
}

zone_area = wide_combined.groupby(field, as_index=False)[["rasterized_area_ha"] + [c for cols in data_area_cols.values() for c in cols]].max()

coverage_rows = []
for source_label, cols in data_area_cols.items():
    df = zone_area[[field, "rasterized_area_ha"] + cols].copy()
    df["data_area_ha"] = df[cols].sum(axis=1)
    df["coverage_status"] = np.select(
        [df["rasterized_area_ha"] == 0, df["data_area_ha"] == 0],
        ["not rasterized", "no input data"],
        default="has data",
    )
    df["source"] = source_label
    coverage_rows.append(df[[field, "rasterized_area_ha", "data_area_ha", "coverage_status", "source"]])

coverage_df = pd.concat(coverage_rows, ignore_index=True)
# Flagged separately to check whether coverage gaps correlate with overlap correction (they don't,
# in every validated run so far -- a reassuring sign that the two mechanisms don't interact badly).
coverage_df["was_overlap_corrected"] = coverage_df[field].isin(overlap_needs_correction_ids)

print(f"Zones with a coverage gap: {(coverage_df['coverage_status'] != 'has data').sum()} of {len(coverage_df)} zone-source pairs: {uu.timestr()}")
display(coverage_df[coverage_df["coverage_status"] != "has data"])

Zones with a coverage gap: 0 of 18 zone-source pairs: 20261014_14_29_04


,raster_id,rasterized_area_ha,data_area_ha,coverage_status,source,was_overlap_corrected


In [17]:
### LULUCF flux vs. component sum reconciliation tab

# Checks that LULUCF's own reported flux matches the sum of its components (veg + mineral soil +
# organic soil). This should hold closely since LULUCF flux is defined as that sum upstream (unlike
# LULUCF's area, which can't be reconstructed from components' separate area totals the same way,
# which is why LULUCF is kept as a full analysis layer rather than dropped as "redundant").
# Repeated per-year values (organic soil's 5-year-interval values, mineral soil's static value) are
# summed as-is across all years combined here, same as everywhere else in this notebook.

component_cols = [
    "veg_gross_emissions_MgCO2e", "veg_gross_removals_MgCO2e", "veg_net_flux_MgCO2e",
    "mineral_soil_gross_loss_MgCO2", "mineral_soil_gross_gain_MgCO2", "mineral_soil_net_change_MgCO2",
    "organic_soil_emissions_total_MgCO2e",
    "LULUCF_gross_emissions_MgCO2e", "LULUCF_gross_removals_MgCO2e", "LULUCF_net_flux_MgCO2e",
]
site_totals = wide_combined.groupby(field, as_index=False)[component_cols].sum()

site_totals["components_emissions_MgCO2e"] = (
    site_totals["veg_gross_emissions_MgCO2e"] + site_totals["mineral_soil_gross_loss_MgCO2"] + site_totals["organic_soil_emissions_total_MgCO2e"]
)
site_totals["emissions_diff_MgCO2e"] = site_totals["components_emissions_MgCO2e"] - site_totals["LULUCF_gross_emissions_MgCO2e"]
site_totals["emissions_pct_diff"] = np.where(
    site_totals["LULUCF_gross_emissions_MgCO2e"] != 0,
    site_totals["emissions_diff_MgCO2e"] / site_totals["LULUCF_gross_emissions_MgCO2e"].abs(), np.nan,
)

site_totals["components_removals_MgCO2e"] = site_totals["veg_gross_removals_MgCO2e"] + site_totals["mineral_soil_gross_gain_MgCO2"]
site_totals["removals_diff_MgCO2e"] = site_totals["components_removals_MgCO2e"] - site_totals["LULUCF_gross_removals_MgCO2e"]
site_totals["removals_pct_diff"] = np.where(
    site_totals["LULUCF_gross_removals_MgCO2e"] != 0,
    site_totals["removals_diff_MgCO2e"] / site_totals["LULUCF_gross_removals_MgCO2e"].abs(), np.nan,
)

site_totals["components_net_MgCO2e"] = (
    site_totals["veg_net_flux_MgCO2e"] + site_totals["mineral_soil_net_change_MgCO2"] + site_totals["organic_soil_emissions_total_MgCO2e"]
)
site_totals["net_diff_MgCO2e"] = site_totals["components_net_MgCO2e"] - site_totals["LULUCF_net_flux_MgCO2e"]
site_totals["net_pct_diff"] = np.where(
    site_totals["LULUCF_net_flux_MgCO2e"] != 0,
    site_totals["net_diff_MgCO2e"] / site_totals["LULUCF_net_flux_MgCO2e"].abs(), np.nan,
)

# drop_duplicates(subset=field) -- same reason as cells 7 and 14 (gdf has multiple rows per field
# value after exploding).
reconciliation_df = site_totals.merge(gdf.drop_duplicates(subset=field)[[field] + output_fields], on=field, how="left")
reconciliation_df = reconciliation_df[
    [field] + output_fields
    + ["veg_gross_emissions_MgCO2e", "mineral_soil_gross_loss_MgCO2", "organic_soil_emissions_total_MgCO2e",
       "components_emissions_MgCO2e", "LULUCF_gross_emissions_MgCO2e", "emissions_diff_MgCO2e", "emissions_pct_diff",
       "veg_gross_removals_MgCO2e", "mineral_soil_gross_gain_MgCO2",
       "components_removals_MgCO2e", "LULUCF_gross_removals_MgCO2e", "removals_diff_MgCO2e", "removals_pct_diff",
       "veg_net_flux_MgCO2e", "mineral_soil_net_change_MgCO2", "organic_soil_emissions_total_MgCO2e",
       "components_net_MgCO2e", "LULUCF_net_flux_MgCO2e", "net_diff_MgCO2e", "net_pct_diff"]
]
# Known cosmetic issue, not yet fixed: organic_soil_emissions_total_MgCO2e gets merged into this
# table twice (once via the emissions comparison, once via the net comparison), so pandas
# auto-suffixes the second occurrence as "organic_soil_emissions_total_MgCO2e.1" in the final Excel
# output. Confirmed byte-identical to the original column -- harmless, just untidy; worth dropping
# the .1 copy next time this cell is touched.

print(f"Reconciliation table: {len(reconciliation_df)} sites: {uu.timestr()}")
display(reconciliation_df)

Reconciliation table: 6 sites: 20261014_14_29_04


,raster_id,Name,veg_gross_emissions_MgCO2e,mineral_soil_gross_loss_MgCO2,organic_soil_emissions_total_MgCO2e,components_emissions_MgCO2e,LULUCF_gross_emissions_MgCO2e,emissions_diff_MgCO2e,emissions_pct_diff,veg_gross_removals_MgCO2e,mineral_soil_gross_gain_MgCO2,components_removals_MgCO2e,LULUCF_gross_removals_MgCO2e,removals_diff_MgCO2e,removals_pct_diff,veg_net_flux_MgCO2e,mineral_soil_net_change_MgCO2,organic_soil_emissions_total_MgCO2e,components_net_MgCO2e,LULUCF_net_flux_MgCO2e,net_diff_MgCO2e,net_pct_diff
0,0,Caatinga,2.735973e+08,1.980018e+08,0.000000e+00,4.715991e+08,4.708930e+08,7.061134e+05,0.001500,-1.427209e+09,-3.188507e+08,-1.746059e+09,-1.745737e+09,-3.225579e+05,-0.000185,-1.153611e+09,-1.208489e+08,0.000000e+00,-1.274460e+09,-1.274844e+09,3.835144e+05,0.000301
1,1,Cerrado,2.593284e+09,8.586830e+08,0.000000e+00,3.451967e+09,3.451051e+09,9.160621e+05,0.000265,-4.848444e+09,-6.331317e+08,-5.481575e+09,-5.480953e+09,-6.219881e+05,-0.000113,-2.255159e+09,2.255513e+08,0.000000e+00,-2.029608e+09,-2.029902e+09,2.940431e+05,0.000145
2,2,Pantanal,2.106389e+08,9.342494e+07,0.000000e+00,3.040638e+08,3.024567e+08,1.607123e+06,0.005314,-2.114273e+08,-2.504575e+07,-2.364730e+08,-2.359815e+08,-4.915449e+05,-0.002083,-7.884115e+05,6.837918e+07,0.000000e+00,6.759077e+07,6.647520e+07,1.115574e+06,0.016782
3,3,Pampa,1.797538e+08,1.051357e+08,0.000000e+00,2.848894e+08,2.847346e+08,1.548010e+05,0.000544,-3.628486e+08,-1.475477e+08,-5.103963e+08,-5.101229e+08,-2.734521e+05,-0.000536,-1.830948e+08,-4.241204e+07,0.000000e+00,-2.255068e+08,-2.253882e+08,-1.186258e+05,-0.000526
4,4,Amazônia,7.855495e+09,1.828490e+09,1.409858e+06,9.685394e+09,9.681542e+09,3.852409e+06,0.000398,-8.467410e+09,-9.290229e+08,-9.396433e+09,-9.394814e+09,-1.618984e+06,-0.000172,-6.119153e+08,8.994668e+08,1.409858e+06,2.889614e+08,2.867283e+08,2.233050e+06,0.007788
5,5,Mata Atlântica,1.752050e+09,9.736655e+08,1.503796e+04,2.725730e+09,2.724987e+09,7.429743e+05,0.000273,-3.664528e+09,-3.989273e+08,-4.063455e+09,-4.063042e+09,-4.138285e+05,-0.000102,-1.912478e+09,5.747383e+08,1.503796e+04,-1.337725e+09,-1.338054e+09,3.291860e+05,0.000246


In [18]:
%%time

print(f"Saving to Excel: {uu.timestr()}")
readme_df = pd.DataFrame({"": readme_lines})

xlsx_path = f'{run_folder}/{out_name}_{today_sparse}.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    readme_df.to_excel(writer, sheet_name="readme", index=False, header=False)
    wide_combined.to_excel(writer, sheet_name="data", index=False)
    reconciliation_df.to_excel(writer, sheet_name="LULUCF_reconciliation", index=False)
print(f"Saved to Excel to {xlsx_path}")

Saving to Excel: 20261014_14_29_04
Saved to Excel to /mnt/c/GIS/AFOLU_flux_model/shapefile_zonal_stats/Brazil_biomes_20261014_141803/Brazil_biomes_20261014.xlsx
CPU times: user 2.04 s, sys: 33.8 ms, total: 2.08 s
Wall time: 2.14 s
